# Phase 2 - Heart Disease Generative AI Advice

**Course:** SWE485 (Selected Topics in Software Engineering)
**Phase:** 2 (Generative AI)

# The Notebook Overview

### Main Objective



The core objective of this generative AI component is to generate a personalised health overview for each patient based on their heart disease classification result and their individual clinical feature values. Given a patient's prediction label and clinical measurements, the system identifies the key patterns present in that patient's data and produces an interpreted overview of their cardiovascular health profile grounded in those individual values, not in generic disease labels.
The overview is designed to translate the model's classification output into plain language, helping the patient understand what their measurements indicate, which values fall outside the healthy range, and what the overall pattern of their data suggests about their cardiovascular status. It may also include general wellness tips where relevant to the patient's specific measurements. This positions the system as a health literacy tool that bridges the gap between a raw classification result and a meaningful, readable interpretation of that assessment.


This objective is shared identically across all four prompt templates. The input is the same for all templates and the output is always a personalised health overview. The four templates differ in their prompting techniques and target user profiles, enabling a broader evaluation of how these factors influence the quality, clarity, and depth of the generated overview.

#### Prompt Inputs

The system uses two primary inputs for each patient assessment: the prediction label and the clinical feature values. Each input plays a specific and deliberate role in enabling the generative model to produce a personalised and clinically grounded interpretation.

**Prediction Label**
The prediction label anchors the entire interpretation. It tells the system
whether the patient has been classified as having heart disease or not,
which determines the overall direction and tone of the generated overview.
Without this label, the system would have no starting point for framing
the response.

**Clinical Feature Values: Before One Hot Encoding**
The feature values are passed to the generative model in their original
categorical and numerical form, before any one hot encoding is applied.
This is a deliberate and important design decision.

One hot encoding is a preprocessing technique designed for machine
learning algorithms that require numerical inputs. It transforms a
categorical feature such as ChestPainType into multiple binary columns
such as ChestPainType_ASY, ChestPainType_ATA, and so on. While this
transformation is necessary for the supervised model to process the
data, it produces a representation that is fragmented, sparse, and
entirely unreadable to a human or a language model. Traditional
preprocessing methods like one hot encoding can inflate the feature
space and reduce interpretability, particularly with high-dimensional
categorical variables (Ali et al., 2026). Passing one hot encoded
features to a generative AI model would result in the model receiving
inputs like ChestPainType_ASY = 1 and ChestPainType_ATA = 0, which
carry no natural language meaning and cannot be used to generate
coherent clinical explanations.

By contrast, passing the original feature value such as
ChestPainType = ASY allows the generative model to understand the
meaning of the input directly, reason about it using its clinical
knowledge, and incorporate it naturally into the generated overview and interpretation.
Large language models are trained on natural language and structured
clinical text, which means they are far better equipped to interpret
and reason about human-readable feature values than binary encoded
representations (Sivarajkumar et al., 2024). This approach ensures
that the interpretation is grounded in clinically meaningful inputs rather
than machine-optimised numerical encodings that obscure the original
meaning of the data.

---

> Ali, A.S., et al. (2026). *Unveiling patterns in clinical data:
> exploring the role of large language models and clustering algorithms.*
> Frontiers in Artificial Intelligence, 9, 1737530.
> https://doi.org/10.3389/frai.2026.1737530

> Sivarajkumar, S., et al. (2024). *An empirical evaluation of prompting
> strategies for large language models in zero-shot clinical natural
> language processing.* JMIR Medical Informatics, 12, e55318.
> https://doi.org/10.2196/55318

#### Handling Output Scenarios

The interpretation generated by the system is not binary. It does not simply say
"you are healthy" or "you have heart disease." Instead, the response is
always anchored to the patient's specific clinical numbers, and the logic
of the interpretation adapts based on two dimensions: the prediction label and
the individual feature values.

**For patients classified as Heart Disease Detected:**
The system identifies which specific clinical features are contributing to
the risk and generates interpretations directly tied to those features. Two patients
with the same label but different clinical values will always receive
different interpretations because it is driven by the individual numbers,
not the label alone.

**For patients classified as No Heart Disease Detected:**
The system does not simply reassure the patient. Instead it evaluates the individual feature values using its embedded clinical knowledge to determine whether they are within healthy or concerning ranges. Two distinct
scenarios are handled:

The first scenario is when the patient's feature values are within healthy
ranges. In this case the interpretation acknowledges the healthy status and
encourages the patient to maintain their current lifestyle and habits,
reinforcing positive health behaviours.

The second scenario is when one or more feature values are borderline,
meaning they are technically within the healthy classification range but
are approaching concerning thresholds. In this case the system generates
a targeted warning for that specific feature, advising the patient to
monitor it and take preventive action before it progresses. This makes
the interpretation genuinely personalised even for patients who are currently
classified as healthy.

This approach reflects the principle that effective health interpretation systems
should go beyond binary classification outputs and provide individuals with
actionable, context-aware guidance tailored to their unique clinical profile
(Li et al., 2024). Furthermore, addressing borderline values
in healthy patients aligns with the broader goal of preventive medicine,
where early identification of risk trajectories is considered more valuable
than waiting for a disease threshold to be crossed (Topol, 2019).

---

> Li, Y.H., Li, Y.L., & Wei, M.Y. (2024). *Innovation and challenges of
> artificial intelligence technology in personalized healthcare.*
> Scientific Reports, 14, 18994.
> https://doi.org/10.1038/s41598-024-70073-7

> Topol, E.J. (2019). *High-performance medicine: The convergence of
> human and artificial intelligence.* Nature Medicine, 25(1), 44–56.
> https://doi.org/10.1038/s41591-018-0300-7


### Shared Prompt Design Principles
 **System and User Prompt Separation**

The prompt structure adopted across all four templates in this project is divided into two distinct parts: a System Instruction and a User Prompt. This design follows the Google Gemini API documentation, which provides a dedicated `system_instruction` parameter that separates the model's persistent behavioural rules from the variable user message (Google, 2024). The system instruction defines the model's role, tone, language level, and constraints, which are rules that must remain fixed for that template regardless of which patient is being assessed. The user prompt then carries the patient-specific data and the task-related description for each patient assessment.

This separation is not merely a technical convention. It reflects a fundamental principle in prompt engineering: that governance rules and task-specific content should never be mixed in the same layer, as doing so risks inconsistency, ambiguity, and unpredictable model behaviour (Tetrate, 2025). When behavioural rules are embedded within the user prompt, they become prone to being overridden, forgotten, or misinterpreted. By contrast, placing them in a dedicated system instruction ensures they are treated with higher priority by the model and remain stable across all requests.

This principle is especially critical in a medical context. In healthcare AI applications, consistency is not a preference but a safety requirement. A system that gives different advice to similar patients, or that drops its safety constraints when the user prompt is complex, poses a direct risk to patient trust and clinical validity. As noted by Gharib et al. (2024), effective prompt design in healthcare requires that the model's behavioural boundaries are clearly established and reliably maintained across all interactions. Similarly, research on prompt engineering in clinical practice emphasises that small changes in prompt structure can significantly impact output quality and safety in medical settings (Cascella et al., 2024). By anchoring all safety rules, role definitions, and tone constraints within the system instruction, each template in this project ensures that no matter what patient data is passed through the user prompt, the model will always respond within the same controlled and medically appropriate framework.

> Google. (2024). *Gemini API documentation: System instructions.*
> Google AI for Developers.
> https://ai.google.dev/gemini-api/docs/system-instructions

> Tetrate. (2025). *System prompts vs user prompts: Design patterns
> for LLM apps.*
> https://tetrate.io/learn/ai/system-prompts-vs-user-prompts

> Gharib, A.M., et al. (2024). *A guide to prompt design: Foundations
> and applications for healthcare simulationists.* Frontiers in Medicine,
> 11, 1504532.
> https://doi.org/10.3389/fmed.2024.1504532

> Cascella, M., Montomoli, J., Bellini, V. and Bignami, E. (2024).
> *Prompt Engineering in Healthcare.* Electronics, 13(15), 2961.
> https://doi.org/10.3390/electronics13152961


**Role Definition**

The system instruction begins by assigning the model a specific professional role as a clinical decision-support assistant specialized in cardiovascular health analysis. This role is intentionally designed to align with the nature of the dataset used in the system, which includes structured patient information such as key health indicators (e.g., cholesterol levels, blood pressure, and other risk-related features).
Establishing a clear role for the model is considered a best practice in prompt engineering. For example, Google’s prompt design guidelines recommend assigning a role to guide the model’s tone, vocabulary, and depth of explanation (Google, 2024). By framing the model as a healthcare-oriented assistant, the generated responses become more consistent, medically relevant, and aligned with real-world clinical reasoning. This ensures that explanations of predicted outcomes are not only technically accurate but also meaningful to end users.
Furthermore, the assigned role encourages the model to interpret predictions in a way that reflects clinical awareness and responsibility, focusing on explaining possible causes, highlighting key contributing factors, and suggesting reasonable next steps when appropriate. At the same time, the system maintains a balance between professionalism and accessibility, ensuring that explanations remain understandable to non-expert users such as patients or general audiences.
This role-based instruction is particularly important given that the system relies on a predictive model trained on patient data stored in the database . By grounding the model’s responses in this role, the system ensures that all generated explanations are context-aware, structured, and aligned with the intended use of the application as an AI-powered health support tool.
Overall, defining the model’s role enhances the quality, reliability, and clarity of the generated explanations, while also supporting the system’s goal of delivering personalized and user-friendly insights based on patient data.


> Google. (2024). *Introduction to prompting.*
> Google AI for Developers.
> https://ai.google.dev/gemini-api/docs/prompting-intro

**Safety Constraints (What the Model Must Never Do)**

Safety constraints are a set of explicit negative rules that define what the model must never do, regardless of the prompting technique or patient data provided. In a cardiovascular health interpretation system, the absence of explicit safety constraints creates real clinical and ethical risk. Guidelines for human-AI interaction recommend that AI systems deployed in sensitive contexts must avoid overconfidence, remain transparent about their limitations, and use language appropriate to the user's emotional state (Amershi et al., 2019). Without an explicit prohibition, a model may produce diagnostic-sounding language because such language is statistically common in the medical text the model was trained on. The constraint must therefore be stated directly in every template, because the model will not infer it from context alone. That said, while safety constraints appear in every template, their specific form differs depending on the intended use case.

> Amershi, S., Weld, D., Vorvoreanu, M., Fourney, A., Nushi, B., Collisson, P., Suh, J., Iqbal, S., Bennett, P., Inkpen, K., Teevan, J., Kikin-Gil, R., and Horvitz, E. (2019). *Guidelines for human-AI interaction.* Proceedings of the 2019 CHI Conference on Human Factors in Computing Systems, 1–13.
> https://doi.org/10.1145/3290605.3300233

**Style Prompting (Tone and Language Control)**

Another component present across all templates is the explicit instruction to control tone and language style. This is formally classified as Style Prompting, a technique that modifies how the model expresses its output rather than what it reasons about or how it structures its response (Sahoo et al., 2024). Style prompting does not change the content, it shapes the way that content is delivered, including the level of formality, the emotional register, and the vocabulary choices the model makes.

In a cardiovascular health interpretation system, tone control has a direct impact on how the patient receives and acts on the information. Research in human-computer interaction confirms that in health-related contexts, warm and conversational language builds significantly more trust than clinical or detached language, and that patients are more likely to engage with recommendations delivered in a tone that feels supportive rather than authoritative (Bickmore and Picard, 2005). Without explicit style instructions, a model with access to medical training data will default to clinical register precise, impersonal, and often alarming to a non-expert reader because that is the dominant tone in the text it learned from.

However, the specific style instructions differ from one template to another based on the intended use case and user profile. The appropriate tone for one user type would be mismatched and potentially counterproductive for another, which is why style prompting must be calibrated individually for each template rather than applied uniformly across the system.

> Sahoo, P., Singh, A.K., Saha, S., Jain, V., Mondal, S. and Chadha, A. (2024). *A systematic survey of prompt engineering in large language models: Techniques and applications.*
> https://arxiv.org/pdf/2406.06608

> Bickmore, T. and Picard, R. (2005). *Establishing and maintaining long-term human-computer relationships.* ACM Transactions on Computer-Human Interaction, 12(2), 293–327.
> https://dl.acm.org/doi/10.1145/1067860.1067867

### Prompt Template Documentation Structure

Each prompt template in this notebook is documented following a structured, stage-based approach. Rather than simply presenting the prompt and its output, each template is walked through seven progressive stages: Define, Justify, Plan, Build, Test, Evaluate and Reflect. This ensures that every design decision is transparent, deliberate, and well-reasoned. The **Define** stage establishes what the prompting technique is and why it suits the medical domain. The **Justify** stage explains the specific use case and the rationale behind choosing this approach. The **Design Explanation** stage outlines the prompt engineering techniques applied. The **Build** stage presents the actual prompt structure. The **Test** stage demonstrates the template with real examples. Finally, the **Evaluate** and **Reflect** stages assess the output quality and acknowledge the template's limitations and assumptions. This progressive structure is applied consistently across all four templates to allow meaningful comparison and analysis.

| # | Phase/Stage | Section | What to Write |
|---|-------------|---------|---------------|
| 1 | | **ID** | A number |
| 2 | Define | **Approach Definition** | Define the prompting technique in simple terms. What is it? How does it work? |
| 3 | Define | **Relation to Medical Field** | Why does this specific technique make sense in a healthcare context? Connect the technique's characteristics to medical needs. |
| 4 | Justify | **Intended Use Case** |  Who is the user? What do they input? What do they need as output? |
| 5 | Justify | **Design Rationale** | Why did we choose this approach over other techniques? What advantages does it have for your specific problem? |
| 6 | Design Explanation | **Prompt Engineering Techniques Applied** | Explain techniques applied in the prompt. Why was each technique included? |
| 7 | Build | **Prompt Structure** | The actual prompt we send to the AI model with placeholders in `{}` for parts that change per user. |
| 8 | Test | **Input / Output Example** | Fill the placeholders with real examples and show the actual AI response you received (3 test cases). |
| 9 | Evaluate | **Evaluation Summary** | Rate the response across the decided criteria. |
| 10 | Reflect | **Assumptions & Limitations** | What must be true for this template to work? Where does it fail or produce weak results? |

### Comparative Analysis Across The 4 Templates

#### Evaluation Framework

Before comparing the four prompt templates, a shared evaluation framework is established. Because each template serves a different user type and applies a different prompting technique, a direct comparison is only meaningful if the criteria used are independent of the specific instructions each template contains.
The criteria below were selected because they reflect output quality dimensions that matter to the patient regardless of which template produced the response. Each template contains many prompt-specific instructions that differ from one template to another. The criteria below represent the dimensions that are universally important across all four templates given the shared deployment context: a hospital mobile application serving patients who have just received a cardiovascular test result.


##### Qualitative Metrics

**1. Edge-Case Handling**

Edge-case handling refers to how the model responds when it receives invalid, extreme, or clinically implausible input values. Since the system accepts patient entered data, there is no guarantee that all inputs will fall within realistic clinical ranges. A user may enter a cholesterol of 0, a resting blood pressure of 500, or leave a value at an implausible extreme. These inputs should not produce a confident interpretation as if the values were legitimate. Instead, the model should recognize that something is unusual about the input and respond in a way that does not mislead the patient. A template that produces a coherent and appropriately cautious response for implausible inputs is more clinically safe than one that generates confident interpretation regardless of what values it receives.

> Coursera Staff (2025). *What Is an Edge Case?* Coursera.
> https://www.coursera.org/articles/edge-case


**2. Instruction Following**

Each of the four templates contains a large number of prompt-specific instructions that vary significantly from one template to another. However, there are three instruction dimensions that are of the highest importance in a medical context. These are selected as the instruction-following criteria because they represent the boundaries that must hold in any AI system operating in a healthcare setting, regardless of the prompting technique used or the user being served. In a non-medical context, failing to follow a prompt instruction may result in a poorly formatted or stylistically inconsistent response, an inconvenience, but not a harm. In a cardiovascular interpretation system deployed in a hospital application, the same failure can directly affect a patient's understanding of their health, their trust in the system, and the decisions they make about seeking care. The three criteria below were therefore selected not because they are the only instructions present, but because they are the ones whose violation carries the highest clinical and ethical consequences.

| Sub-category | Description | Why It Matters in Our Context |
|---|---|---|
| Personalization | Does the output adapt to the specific patient's clinical values rather than producing a generic label-driven response? | Two patients with the same prediction label but different values must receive different responses. Patient-centered medicine holds that the primary objective of any health system is to improve outcomes for the individual patient in everyday clinical practice, taking into account their specific values and circumstances rather than treating all patients with the same condition identically (Sacristán, 2013). A system that responds to the prediction label alone, ignoring the individual clinical values that produced it, directly contradicts this principle. |
| Context | Does the model maintain the clinical and patient context established in the prompt — including the deployment setting, the user profile, and the nature of the input as cardiovascular test results without drifting into unrelated or inappropriate territory? | In a hospital mobile application, context drift, where the model begins responding as if it were a general chatbot rather than a cardiovascular interpretation tool, directly undermines clinical credibility and patient trust. |
| Safety — No Diagnosis | Does the model avoid stating or implying a diagnosis in any form, and does it frame all findings as possibilities rather than confirmed facts? | This is the most critical safety boundary across all four templates. The system is not a licensed clinical instrument. A response that presents findings as a confirmed diagnosis could cause the patient to stop seeking professional care, act on incorrect information, or experience serious psychological harm.|

> Sacristán, J.A. (2013). *Patient-centered medicine and patient-oriented research: improving health outcomes for individual patients.* BMC Medical Informatics and Decision Making, 13, 6.
> https://doi.org/10.1186/1472-6947-13-6


##### Quantitative Metrics
**1. Flesch-Kincaid Readability Score**

| Description | Why It Matters in Our Context | Scoring Rule |
|---|---|---|
| Measures text readability based on sentence length and word complexity| All four templates serve non-expert patients reading on a mobile screen. A response that is clinically accurate but linguistically inaccessible fails its intended audience. Research confirms that health literacy is a significant barrier to patient engagement, and that simpler language directly improves comprehension and adherence (Zolnierek and DiMatteo, 2009). | Target: 60–70, which corresponds to plain English accessible to a general adult audience. Higher is better. Computed using the `textstat` library in Python on the interpretation paragraph. |

> Zolnierek, K.B.H. and DiMatteo, M.R. (2009). *Physician communication and patient adherence to treatment: A meta-analysis.* Medical Care, 47(8), 826–834.
> https://doi.org/10.1097/MLR.0b013e31819a5acc

The following code calculates the readability score for each generated output and then computes the average score across all test cases. This helps provide a clear quantitative view of how easy the responses are to read and whether they fall within the intended readability range.


In [80]:
# Natural language toolkit
import nltk
# SSL handling for downloads
import ssl
# Readability score calculation
import textstat

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# Download dictionary
nltk.download('cmudict')


# Evaluate readability scores for generated outputs
def evaluate_readability(outputs):
    print("Flesch-Kincaid Readability Scores:")
    print("-" * 50)

    scores = []
    results = {}

    for name, text in outputs.items():
        score = textstat.flesch_reading_ease(text)
        scores.append(score)
        results[name] = round(score, 2)
        print(f"{name}: {round(score, 2)}")

    average = round(sum(scores) / len(scores), 2) if scores else 0

    print(f"\nTemplate Average: {average}")
  

    return {
        "scores_by_case": results,
        "average_score": average,
        
    }

[nltk_data] Downloading package cmudict to /Users/loba/nltk_data...
[nltk_data]   Package cmudict is already up-to-date!


**2. Domain Keyword Density**

| Description | Why It Matters in Our Context | Scoring Rule |
|---|---|---|
| The percentage of cardiovascular-relevant terms in the output relative to total word count. | A response that contains too few domain terms may be too vague to be clinically meaningful. A response with too many may be too technical for a non-expert patient. The density score measures whether the model strikes the right balance of clinical grounding and accessibility for each template's intended user. | Density = domain keyword count / total word count × 100. The appropriate range is assessed relative to each template's intended user. A lower density is expected and appropriate for the zero-shot anxious patient template, while a higher density is expected for the chain-of-thought health-literate patient template. |

The domain keyword reference list was constructed by combining the
cardiovascular terminology from the Texas Heart Institute
Cardiovascular Glossary with additional terms identified by
inspecting all four template outputs. The final reference list was
saved as `cardiovascular_domain_reference_list.txt` and used as
the basis for counting domain keyword occurrences in each output.


> Texas Heart Institute. (2024). *Cardiovascular Glossary A–Z.*
> https://www.texasheart.org/heart-health/heart-information-center/topics/a-z/

In [81]:
# Load cardiovascular domain keywords from reference file
with open("cardiovascular_domain_reference_list.txt.txt", "r") as f:
    cv_keywords = set(line.strip().lower() for line in f if line.strip())


# Function to count domain keyword occurrences in a text
def count_domain_keywords(text):
    words = text.lower().split()
    count = sum(1 for word in words if word.strip(".,!?;:'\"()") in cv_keywords)
    return count



# Evaluate domain keyword density for generated outputs
def evaluate_keyword_density(outputs):
    print("Domain Keyword Density:")
    print("-" * 50)

    densities = []
    results = {}

    for name, text in outputs.items():
        total_words = len(text.split())
        domain_count = count_domain_keywords(text)
        density = round(domain_count / total_words * 100, 2) if total_words > 0 else 0

        densities.append(density)
        results[name] = {
            "total_words": total_words,
            "domain_keywords": domain_count,
            "density": density
        }

        print(f"{name}:")
        print(f"  Total Words       : {total_words}")
        print(f"  Domain Keywords   : {domain_count}")
        print(f"  Density           : {density}%")
        print()

    average = round(sum(densities) / len(densities), 2) if densities else 0
    print(f"Template Average Density: {average}%")

    return {
        "density_by_case": results,
        "average_density": average
    }

**3. Medical Accuracy**

| Description | Why It Matters in Our Context | Scoring Rule |
|---|---|---|
| The factual correctness of the clinical statements the model makes, assessed against established cardiovascular guidelines. | An interpretation that contains a factually incorrect clinical statement could mislead a patient and directly contradict guidance their doctor provides. In a medical context, factual accuracy is not a quality preference but a safety requirement. | Each clinical claim in the output is verified against established cardiovascular reference guidelines. Accuracy rate = correct claims / total claims × 100. |

The following cardiovascular reference guidelines will consider whether the interpretation of RestingBP, Cholesterol, FastingBS, MaxHR, and Oldpeak is factually correct and consistent with accepted cardiovascular understanding. In addition, ChestPainType, RestingECG, ExerciseAngina, and STSlope will be evaluated based on findings from relevant cardiovascular research and clinical analysis, to determine whether their described significance and relationship to heart disease risk are medically accurate and aligned with established evidence.

*RestingBP (Blood Pressure)*
| Normal | Elevated (Borderline) | High |
|---|---|---|
| < 120 | 120–139 | ≥ 140 |

*Cholesterol*
| Normal | Elevated (Borderline) | High |
|---|---|---|
| < 200 | 200-239 | ≥ 240 |

*FastingBS*
| Normal | Elevated (Borderline) | Diabetes |
|---|---|---|
| 70–99 | 100–125 | ≥ 126 |

*MaxHR*

Predicted MaxHR = 211 - (0.64 * Age)

Attainment % = (Observed MaxHR / Predicted Max HR) × 100
| Low | Normal | High |
|---|---|---|
| < 70% | 70%–85% | > 85% |

*Oldpeak*
| Low | Elevated (Borderline) | High |
|---|---|---|
| < 1 | 1–2 | ≥ 2 |

> Irish Heart Foundation. (n.d.). Blood pressure – what you need to know. Irish Heart Foundation.
> https://irishheart.ie/how-to-keep-your-heart-healthy/blood-pressure/

> Katella, K. (2026). Your Cholesterol Numbers: The Good, the Bad, the Triglycerides. Yale Medicine.
> https://www.yalemedicine.org/news/your-cholesterol-numbers-the-good-the-bad-the-triglycerides

> MedlinePlus. (2025). Blood sugar test. U.S. National Library of Medicine.
> https://medlineplus.gov/ency/article/003482.htm

> NTNU CERG. (n.d.). Heart Rate Calculator. Norwegian University of Science and Technology.
> https://www.ntnu.edu/cerg/hrmax

> Darrab, S., Broneske, D., & Saake, G. (2024). Exploring the predictive factors of heart disease using rare association rule mining.
> https://www.nature.com/articles/s41598-024-69071-6

**4. Passive Voice Rate**
| Description | Why It Matters in Our Context | Scoring Rule |
|---|---|---|
| The percentage of sentences in the output that use passive voice construction. | Active voice places the patient at the centre of the recommendation "you should monitor your blood pressure" rather than "your blood pressure should be monitored." According to the American Medical Association health literacy recommendations, all patient education materials should be written using active voice to improve patient understanding and adherence to treatment plans — a standard that is particularly relevant in cardiovascular health contexts where patient comprehension directly affects outcomes (Tucker, 2024). A high passive voice rate suggests the output reads more like a clinical report than a patient interpretation.| Passive voice rate = passive sentences / total sentences × 100. Measured using the `nsubjpass` dependency tag in `spacy`. Lower is better. |

> Tucker, C.A. (2024). *Promoting personal health literacy through readability, understandability, and actionability of online patient education materials.* Journal of the American Heart Association, 13(14).

> https://doi.org/10.1161/JAHA.124.033916

The following code calculates the passive voice rate for each generated output by identifying how many sentences are written in passive voice relative to the total number of sentences. It then computes the average passive voice rate across all test cases, which helps evaluate whether the responses use a more direct and patient-appropriate writing style.

In [3]:
!python3 -m spacy download en_core_web_sm

/opt/anaconda3/bin/python3: No module named spacy


In [82]:
# Natural language processing library
import spacy

# Load the English language model
nlp = spacy.load("en_core_web_sm")


# Calculate passive voice rate for one text
def passive_voice_rate(text):
    doc = nlp(text)
    sentences = list(doc.sents)
    passive_count = 0

    for sent in sentences:
        for token in sent:
            if token.dep_ == "nsubjpass":
                passive_count += 1
                break

    rate = round(passive_count / len(sentences) * 100, 2) if sentences else 0
    return rate, passive_count, len(sentences)


# Evaluate passive voice rate for generated outputs
def evaluate_passive_voice(outputs):
    print("Passive Voice Rate:")
    print("-" * 50)

    rates = []
    results = {}

    for name, text in outputs.items():
        rate, passive_count, total_sents = passive_voice_rate(text)
        rates.append(rate)

        results[name] = {
            "total_sentences": total_sents,
            "passive_sentences": passive_count,
            "passive_rate": rate
        }

        print(f"{name}:")
        print(f"  Total Sentences  : {total_sents}")
        print(f"  Passive Sentences: {passive_count}")
        print(f"  Passive Rate     : {rate}%")
        print()

    average = round(sum(rates) / len(rates), 2) if rates else 0
    print(f"Template Average Passive Voice Rate: {average}%")

    return {
        "passive_voice_by_case": results,
        "average_passive_rate": average
    }

# Generative AI Model Setup

In [106]:
import pickle

# Load the best model saved from Phase 1 (XGBoost model)
with open("Supervised_Learning/models/best_xgb.pkl", "rb") as f:
    model = pickle.load(f)

In [79]:
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import StandardScaler
from sklearn.preprocessing import RobustScaler

#Method to get prediction from the model, this will be used in the prompt for Gemini

#Step 1:
# Fit scalers on the same training data statistics
# Age → StandardScaler | RestingBP → RobustScaler | MaxHR → StandardScaler | Oldpeak → RobustScaler
raw_df =  pd.read_csv("Dataset/preprocessed_heart_data.csv")

#WE NEED TO CHECK THIS!!
age_scaler      = StandardScaler().fit(raw_df[['Age']])
resting_bp_scaler = RobustScaler().fit(raw_df[['RestingBP']])
max_hr_scaler   = StandardScaler().fit(raw_df[['MaxHR']])
oldpeak_scaler  = RobustScaler().fit(raw_df[['Oldpeak']])

def get_prediction(inputs):

    #Step 2: Cholesterol → category then drop raw value
    chol = inputs["Cholesterol"]
    chol_bins   = [0, 200, 240, np.inf]
    chol_labels = ["Desirable", "Borderline High", "High"]
    chol_cat    = pd.cut([chol], bins=chol_bins, labels=chol_labels, right=False)[0]

    # Step 3:Build encoded row
    encoded = {
        "Age":                              inputs["Age"],
        "Sex":                              1 if inputs["Sex"] == "Male" else 0,
        "RestingBP":                        inputs["RestingBP"],
        "FastingBS":                        1 if inputs["FastingBS"] > 120 else 0,
        "MaxHR":                            inputs["MaxHR"],
        "ExerciseAngina":                   1 if inputs["ExerciseAngina"] == "Yes" else 0,
        "Oldpeak":                          inputs["Oldpeak"],
        # ChestPainType one-hot
        "ChestPainType_ASY":                1 if inputs["ChestPainType"] == "ASY" else 0,
        "ChestPainType_ATA":                1 if inputs["ChestPainType"] == "ATA" else 0,
        "ChestPainType_NAP":                1 if inputs["ChestPainType"] == "NAP" else 0,
        "ChestPainType_TA":                 1 if inputs["ChestPainType"] == "TA"  else 0,

        # RestingECG one-hot
        "RestingECG_LVH":                   1 if inputs["RestingECG"] == "LVH"    else 0,
        "RestingECG_Normal":                1 if inputs["RestingECG"] == "Normal" else 0,
        "RestingECG_ST":                    1 if inputs["RestingECG"] == "ST"     else 0,

        # ST_Slope one-hot
        "ST_Slope_Down":                    1 if inputs["STSlope"] == "Down" else 0,
        "ST_Slope_Flat":                    1 if inputs["STSlope"] == "Flat" else 0,
        "ST_Slope_Up":                      1 if inputs["STSlope"] == "Up"   else 0,

        # Chol_category one-hot
        "Chol_category_Desirable":          1 if chol_cat == "Desirable"       else 0,
        "Chol_category_Borderline High":    1 if chol_cat == "Borderline High" else 0,
        "Chol_category_High":               1 if chol_cat == "High"            else 0,
    }

    df = pd.DataFrame([encoded])

    # Step 4: Apply same scalers from Phase 1
    df["Age"]       = age_scaler.transform(df[["Age"]])
    df["RestingBP"] = resting_bp_scaler.transform(df[["RestingBP"]])
    df["MaxHR"]     = max_hr_scaler.transform(df[["MaxHR"]])
    df["Oldpeak"]   = oldpeak_scaler.transform(df[["Oldpeak"]])

    result = model.predict(df)
    return "Heart Disease" if result[0] == 1 else "No Heart Disease"

In [3]:
#JUST AN EXAMPLE, EACH ONE OF US WILL DO LIKE THIS WHEN HER PROMPT IS READY, SHE WILL PREPARE THE INPUT, THEN SEND TO OUR MODEL TO GET THE
#PREDICTION THAT WILL BE SEND WITHIN THE PROMPT
inputs = {
    "Age":              55,
    "Sex":              "M",
    "ChestPainType":  "ATA",
    "RestingBP":       140,
    "Cholesterol":      240,
    "FastingBS":       1,
    "RestingECG":      "Normal",
    "MaxHR":           150,
    "ExerciseAngina":  "Y",
    "Oldpeak":          2.3,
    "STSlope":         "Flat"
}

inputs["prediction"] = get_prediction(inputs)
print(inputs["prediction"])

Heart Disease


### Generative AI setup

In [107]:
#Imports & Client Setup
import os
from google import genai
from dotenv import load_dotenv, find_dotenv

load_dotenv(override=True)  # override=True forces it to reload
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)
MODEL = "gemini-2.5-flash"

In [108]:
#Shared Method
def call_gemini(system_prompt, user_prompt):
    response = client.models.generate_content(
        model=MODEL,
        contents=user_prompt,
        config={"system_instruction": system_prompt}
    )
    return response.text

In [6]:
#JUST AN EXAMPLE, EACH ONE OF US WILL DO LIKE THIS WHEN HER PROMPT IS READY, SHE WILL PREPARE THE INPUT, THE USER PROMPT, THE SYSTEM PROMPT
#THEN CALL THIS METHOD THAT IS ALREADY PREPARED FOR YOU GUYS
inputs = {
    "prediction":       "No Heart Disease",
    "Age":              55,
    "Sex":              "Male",
    "ChestPainType":  "ATA",
    "RestingBP":       140,
    "Cholesterol":      240,
    "FastingBS":       1,
    "RestingECG":      "Normal",
    "MaxHR":           150,
    "ExerciseAngina":  "Yes",
    "Oldpeak":          2.3,
    "STSlope":         "Flat"
}

SYSTEM_T1 = """You are a medical assistant that helps explain heart disease risk to patients."""

USER_T1 = """A patient has been assessed and the result is: {prediction}.
Patient details: Age={Age}, Sex={Sex}, Chest Pain={ChestPainType}, Resting BP={RestingBP},
Cholesterol={Cholesterol}, Fasting BS={FastingBS}, Resting ECG={RestingECG},
Max HR={MaxHR}, Exercise Angina={ExerciseAngina}, Oldpeak={Oldpeak}, ST Slope={STSlope}.
Please provide a brief and simple explanation of this result."""

result_t1 = call_gemini(SYSTEM_T1, USER_T1.format(**inputs))
print(result_t1)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

# Generated Knowledge Prompting

 ### Template ID
 1

---
### Approach Definition

Generated Knowledge Prompting is a prompt engineering technique introduced
by Liu et al. (2022) that instructs the model to first generate relevant
knowledge about the topic before producing its final response. Rather than
asking the model to answer directly, the prompt is structured in two
internal stages within a single request: the model first surfaces and
organises what it knows about the subject, and then uses that
self-generated knowledge as the foundation for its final output.

The core idea behind this technique is that large language models contain
vast amounts of knowledge from their training, but do not always surface
the most relevant information when asked to respond directly. By explicitly
instructing the model to generate knowledge first, the prompt forces a
broader and more deliberate retrieval of relevant information before any
conclusion is produced. This generated knowledge then sits within the
model's context during the response generation phase, ensuring that the final
response is grounded in a richer and more organised understanding of the
topic (Liu et al., 2022). This approach does not require access to an
external knowledge base or any task-specific training, making it flexible
and practical for a wide range of applications.

> Liu, J., Liu, A., Lu, X., Welleck, S., West, P., Le Bras, R., Choi, Y.,
> and Hajishirzi, H. (2022). *Generated Knowledge Prompting for Commonsense
> Reasoning.* Proceedings of the 60th Annual Meeting of the Association for
> Computational Linguistics, 3154–3169.
> https://doi.org/10.18653/v1/2022.acl-long.225


---

### Relation to Medical Field

In a medical context, generating knowledge before producing a response is
particularly valuable because clinical reasoning is rarely a direct
input-to-output process. A clinician examining a patient's cardiovascular
data does not immediately jump to a conclusion. They first consider what
each measurement means, how it relates to known clinical patterns, and what
risks it might indicate, before forming an assessment and recommending
next steps. Generated Knowledge Prompting mirrors this interpretive process
by instructing the model to first surface and organise its clinical
knowledge about the patient's specific feature values before producing
the final interpretation.

This is especially important in cardiovascular health, where the clinical
significance of a measurement depends heavily on context. A resting blood
pressure of 145 mmHg, for example, cannot be interpreted in isolation. The
model needs to draw on its knowledge of what that value means, how it
relates to cardiovascular risk, and how it interacts with other features
in the patient's profile before it can generate interpretation that is genuinely
meaningful. By explicitly instructing the model to generate this contextual
knowledge first, the prompt ensures that the final interpretation is grounded in
a deliberate clinical reasoning step rather than a surface level response.


---


### Intended Use Case

 **1- Deployment Environment**

The system is planned to be deployed as an interpretation tool within a hospital
mobile application. Patients access it from their personal smartphones
after obtaining their clinical data, either from a recent cardiovascular
test or by completing one at the hospital. The patient enters their
clinical values into the tool and receives a personalized interpretation
directly on the same screen.

 **2- The User**

The primary user of this template is a **first-time patient** who is
encountering the system and their clinical results for the very first
time. This patient arrives with no frame of reference for what their
numbers mean. A resting blood pressure of 145 mmHg is just a number to
them. An Oldpeak value of 2.3 carries no weight without context. A
cholesterol category label means nothing without knowing what it implies
for their health. Providing interpretation to this patient without first
establishing what their values mean would produce a response they cannot
connect to their own situation, and interpretation that cannot be connected to
is an interpretation that will not be acted on.

***Medical background:*** No medical background is assumed. The gap
between what the patient's values say and what they mean to that patient
is treated as the central design constraint of this template. Before any
recommendation can land meaningfully, the patient needs enough
foundational understanding to connect the interpretation back to their own
situation rather than receiving it as an instruction they must simply
trust.

***Emotional state:*** First-time users often arrive uncertain and quietly
apprehensive. They may not know what they are looking for or what a
concerning result would even look like. The tone of the interpretation must
therefore be steady and educational rather than clinical or alarming.
The goal is not to reassure unconditionally or to alarm unnecessarily,
but to leave the patient feeling genuinely informed and oriented enough
to take the next right step.

**3- Inputs**

3.1- The patient heart disease classification result

3.2- The patient clinical feature values


**4- Output**

A single interpretation paragraph written in plain, accessible language.
Although the model generates clinical knowledge internally as part of
its knowledge generation process, none of this internal knowledge generation is
visible to the patient. The output reads as a clean, direct interpretation,
but because it was built on top of a deliberate knowledge generation
step, it is more grounded in the patient's specific values and more
educationally coherent than a direct response would be for someone
encountering their clinical data for the first time.


---

### Design Rationale

**Advantage 1 — Grounded knowledge generation before response:**
Generated Knowledge Prompting instructs the model to first surface and
organise relevant knowledge about the input before producing its final
response, ensuring the output is grounded in an explicit knowledge generation step
rather than a direct leap from input to answer (Liu et al., 2022). In a
heart disease interpretation system, this matters because the clinical
significance of a patient's feature values cannot be determined without
first interpreting what those values mean in a cardiovascular context.
A model that jumps directly to interpretation without first generating knowledge about the
patient's specific measurements risks producing generic or misaligned
recommendations that do not reflect the patient's actual profile.

**Advantage 2 — Improved contextual relevance for individual patients:**
By generating knowledge that is specific to the input provided, the
technique ensures that the final response is tailored to the individual
patient rather than relying on broad generalisations (Liu et al., 2022).
In a heart disease interpretation system where two patients can share the same
prediction label but have entirely different clinical profiles, this
matters directly. A patient with a borderline resting blood pressure and
a flat ST slope requires fundamentally different explanation and interpretation from a patient
with a normal blood pressure and an asymptomatic chest pain type, even
if both are classified as No Heart Disease Detected. Generated Knowledge Prompting ensures the model generates contextual
knowledge about the specific values present before writing the interpretation,
producing a response that reflects the individual rather than the label.

**Advantage 3 — No dependency on predefined examples:**
Generated Knowledge Prompting does not require carefully crafted
input-output examples to guide the model's behaviour. The model draws
on its own clinical knowledge to reason about the patient's data, making
the technique flexible and robust across the full range of possible
patient profiles (Liu et al., 2022). In a heart disease interpretation system
where patient feature values vary continuously across a wide range,
providing representative examples that cover all meaningful combinations
is not practical. Generated Knowledge Prompting eliminates this
dependency by allowing the model to generate the contextual knowledge
it needs dynamically from the input itself.


> Liu, J., Liu, A., Lu, X., Welleck, S., West, P., Le Bras, R., Choi, Y.,
> and Hajishirzi, H. (2022). *Generated Knowledge Prompting for Commonsense
> Reasoning.* Proceedings of the 60th Annual Meeting of the Association for
> Computational Linguistics, 3154–3169.
> https://doi.org/10.18653/v1/2022.acl-long.225

---



### Prompt Engineering Techniques Applied

#### Role Assignment

*You are a clinical decision-support assistant specialised in cardiovascular health.*

The system instruction opens by assigning the model a specific professional role as a clinical decision-support assistant specialised in cardiovascular health. This role was deliberately chosen to align with the nature of the system, which operates on cardiovascular screening data and produces patient-facing insights about heart health. In the context of Generated Knowledge Prompting, the role assignment is particularly important because the quality of the knowledge the model generates internally in Stage 1 depends directly on the domain it believes it is operating in. By assigning a cardiovascular specialist role, the model is guided to surface clinical knowledge that is specific to cardiovascular health rather than drawing on broad general medical knowledge. This ensures that the internally generated knowledge is focused, relevant, and clinically appropriate before it is used to produce the final insight.

#### User Context Priming

*The patient receiving this insight is a first-time user who has never seen or interpreted their clinical data before. They have no frame of reference for what their values mean.*

The system instruction explicitly describes the target user as a first-time patient with no frame of reference for their clinical values. This context primes the model to calibrate the depth, length, and tone of its response appropriately. Without this context, the model may produce responses that are either too technical for a first-time patient or too generic to be meaningful (Cascella et al., 2024).

> Cascella, M., Montomoli, J., Bellini, V. and Bignami, E. (2024). *Prompt Engineering in Healthcare.* Electronics, 13(15), 2961.
> https://doi.org/10.3390/electronics13152961

#### Conciseness Constraint

*Do not over-explain. Do not list every value. Focus only on the most relevant values and explain them briefly and clearly. The goal is to leave the patient feeling informed and oriented, not overwhelmed.*

The system instruction explicitly instructs the model not to over-explain and to focus only on the most relevant values. This constraint is particularly important for a first-time patient who may feel overwhelmed by too much information. Keeping the output concise and focused ensures the patient leaves the interaction feeling informed and oriented rather than confused (Cascella et al., 2024).

> Cascella, M., Montomoli, J., Bellini, V. and Bignami, E. (2024). *Prompt Engineering in Healthcare.* Electronics, 13(15), 2961.
> https://doi.org/10.3390/electronics13152961

#### Tone Specification

*Use a calm, grounding, and educational tone.*

The system instruction explicitly specifies the tone the model must maintain throughout the response: calm, grounding, and educational. In a medical context, tone is as important as content. A response that is accurate but alarming or cold can cause unnecessary distress for a first-time patient. By specifying the tone explicitly, the prompt ensures the model communicates in a way that is appropriate for someone encountering their clinical data for the very first time.

#### Value Grounding

*Be grounded in the patient's specific values, not in generic disease labels.*

The system instruction explicitly instructs the model to ground its response in the patient's specific clinical values rather than generic disease labels. This ensures that every patient receives a response that reflects their individual numbers, not a template response triggered by the prediction label alone. This is essential for personalisation and is directly aligned with the objective of the system (Liu et al., 2022).

> Cascella, M., Montomoli, J., Bellini, V. and Bignami, E. (2024). *Prompt Engineering in Healthcare.* Electronics, 13(15), 2961.
> https://doi.org/10.3390/electronics13152961

#### Conditional Response Logic

*If the result is Heart Disease Detected: focus on explaining which values are contributing to the result and what they mean. If the result is No Heart Disease Detected and all values are healthy: reassure the patient and explain which specific values are within healthy ranges and what each one means for their cardiovascular health. If the result is No Heart Disease Detected but one or more values are borderline: acknowledge the assessment result but explain the borderline values and what they could mean if left unaddressed*

The system instruction defines three distinct response scenarios based on the prediction label and individual feature values: Heart Disease Detected, No Heart Disease with all healthy values, and No Heart Disease with borderline values. Each scenario is given specific and detailed instructions for how the model should respond, ensuring the output is always contextually appropriate regardless of which scenario the patient falls into.

#### Safety Constraints

*Never use clinical jargon without explanation. Never state or imply a diagnosis. Always refer to the assessment result or the clinical indicators instead. Recommend consulting a medical professional only when it is clinically necessary.*

The system instruction includes explicit safety constraints that the model must follow in every response. The model must never state or imply a diagnosis, must never use clinical jargon without explanation, and must only recommend consulting a medical professional when it is clinically necessary. These constraints ensure the system operates safely and responsibly within a medical context where an inappropriate response could directly impact patient trust and wellbeing.

### Prompt Structure

In [ ]:
SYSTEM_T1 = """You are a clinical decision-support assistant specialised in cardiovascular health.
You communicate in plain, accessible language suitable for patients with no medical background.

The patient receiving this insight is a first-time user who has never seen
or interpreted their clinical data before. They have no frame of reference
for what their values mean. Your insight must therefore be concise and
focused. Do not over-explain. Do not list every value. Focus only on the
most relevant values and explain them briefly and clearly. The goal is to
leave the patient feeling informed and oriented, not overwhelmed.

You follow a two-stage internal process for every patient assessment:

STAGE 1 — GENERATE CLINICAL KNOWLEDGE (INTERNAL ONLY — DO NOT SHOW TO PATIENT):
Before writing any insight, you must first internally generate your clinical
knowledge about each of the patient's values. For each feature provided, establish
what that value means, whether it is healthy, borderline, or concerning, and what
its implications are for cardiovascular health. This stage must never appear in
your response. It is a silent knowledge generation step only.

STAGE 2 — WRITE THE INSIGHT (THIS IS THE ONLY THING YOU OUTPUT):
Using the knowledge you generated in Stage 1, write a structured personalised
insight for the patient. The insight must follow this exact format:

[One cohesive paragraph that introduces the assessment result, explains the
most relevant values and what they indicate, includes one or two practical
tips directly related to those values, and recommends consulting a medical
professional only if clinically necessary.]

The insight must also follow these rules:
- The core purpose is interpretation. Brief practical tips are acceptable
  but detailed medical recommendations are not.
- Be grounded in the patient's specific values, not in generic disease labels
- If the result is Heart Disease Detected: focus on explaining which values
  are contributing to the result and what they mean
- If the result is No Heart Disease Detected and all values are healthy:
  reassure the patient and explain which specific values are within healthy
  ranges and what each one means for their cardiovascular health. Help them
  understand why their results are reassuring, not just that they are.
- If the result is No Heart Disease Detected but one or more values are borderline:
  acknowledge the assessment result but explain the borderline values and what
  they could mean if left unaddressed
- Use a calm, grounding, and educational tone
- Never use clinical jargon without explanation
- The prediction label must always be referenced in the response but
  treated as a probabilistic assessment, not a clinical certainty.
  Present it as what the assessment suggests, not as a confirmed fact.
- Recommend consulting a medical professional only when it is clinically
  necessary, such as when values are concerning or borderline. Do not
  include this recommendation if all values are healthy and reassuring.
- NO headers, NO bullet points, NO section titles, NO numbered sections


YOUR RESPONSE MUST CONTAIN ONLY ONE SINGLE PARAGRAPH.
NOTHING ELSE."""


USER_T1 = """Patient Assessment Result: {prediction}

Patient Clinical Values:
- Age: {Age} years
- Sex: {Sex}
- Chest Pain Type: {ChestPainType}
- Resting Blood Pressure: {RestingBP} mmHg
- Cholesterol: {Cholesterol} mg/dL
- Fasting Blood Sugar : {FastingBS} 
- Resting ECG: {RestingECG}
- Maximum Heart Rate: {MaxHR} bpm
- Exercise-Induced Angina: {ExerciseAngina}
- Oldpeak (ST Depression): {Oldpeak}
- ST Slope: {STSlope}

First generate your internal clinical knowledge about each feature silently,
then write the structured personalised insight for this patient
following the exact format specified. Remember that the prediction label
is a probabilistic result, not a confirmed diagnosis. Frame it accordingly."""

### Input/Output Examples

#### Test case 1

In [6]:
# Test Case 1: Heart Disease Detected
inputs_1 = {
    "Age":              55,
    "Sex":              "Male",
    "ChestPainType":    "ATA",
    "RestingBP":        140,
    "Cholesterol":      240,
    "FastingBS":        160,
    "RestingECG":       "Normal",
    "MaxHR":            150,
    "ExerciseAngina":   "Yes",
    "Oldpeak":          2.3,
    "STSlope":          "Flat"
}
inputs_1["prediction"] = get_prediction(inputs_1)

from IPython.display import display, Markdown
print("Test Case 1 — Heart Disease Detected")
result_1 = call_gemini(SYSTEM_T1, USER_T1.format(**inputs_1))
display(Markdown(result_1))


Test Case 1 — Heart Disease Detected


Your assessment suggests the presence of heart disease, which is indicated by several key values. Your resting blood pressure of 140 mmHg and cholesterol level of 240 mg/dL are higher than recommended, indicating increased strain on your heart and arteries. Additionally, your fasting blood sugar of 160 mg/dL is elevated, which can impact your blood vessels over time. You also reported typical chest pain, especially during exercise, along with significant changes on your exercise ECG, specifically a 2.3 mm ST depression and a flat ST slope, which are strong indicators of reduced blood flow to your heart during physical activity. Taking steps to manage your blood pressure, cholesterol, and blood sugar, such as adopting a balanced diet and regular physical activity, can be beneficial. Given these results, it's important to consult with a medical professional to discuss these findings further and explore appropriate next steps for your heart health.

#### Test case 2

In [7]:
# Test Case 2: No Heart Disease Detected — All Healthy Values
inputs_2 = {
    "Age":              35,
    "Sex":              "Female",
    "ChestPainType":    "NAP",
    "RestingBP":        110,
    "Cholesterol":      180,
    "FastingBS":        80,
    "RestingECG":       "Normal",
    "MaxHR":            175,
    "ExerciseAngina":   "No",
    "Oldpeak":          0.0,
    "STSlope":          "Up"
}
inputs_2["prediction"] = get_prediction(inputs_2)

from IPython.display import display, Markdown
print("Test Case 2 — No Heart Disease: All Healthy Values")
result_2 = call_gemini(SYSTEM_T1, USER_T1.format(**inputs_2))
display(Markdown(result_2))

Test Case 2 — No Heart Disease: All Healthy Values


Your assessment suggests no heart disease, which is a very positive indication for your cardiovascular health. Your blood pressure of 110 mmHg is well within a healthy range, meaning your heart and blood vessels are not under undue strain. Similarly, your cholesterol level of 180 mg/dL and fasting blood sugar of 80 mg/dL are both healthy, which helps prevent blockages in your arteries and supports overall well-being. Additionally, your resting ECG is normal, and during exercise, you showed no signs of angina or concerning changes like ST depression, indicating your heart functions well under stress. Continuing to maintain these healthy habits through a balanced diet and regular physical activity will further support your excellent heart health.

#### Test case 3

In [8]:

# Test Case 3: No Heart Disease Detected — Borderline Values
inputs_3 = {
    "Age":              45,
    "Sex":              "Male",
    "ChestPainType":    "NAP",
    "RestingBP":        128,
    "Cholesterol":      210,
    "FastingBS":        100,
    "RestingECG":       "Normal",
    "MaxHR":            160,
    "ExerciseAngina":   "No",
    "Oldpeak":          0.8,
    "STSlope":          "Up"
}
inputs_3["prediction"] = get_prediction(inputs_3)


from IPython.display import display, Markdown
print("Test Case 3 — No Heart Disease: Borderline Values")
result_3 = call_gemini(SYSTEM_T1, USER_T1.format(**inputs_3))
display(Markdown(result_3))

Test Case 3 — No Heart Disease: Borderline Values


Based on your assessment, it suggests no heart disease was detected, which is generally reassuring. We observed that your resting ECG is normal, you don't experience chest pain indicating angina, and your blood sugar is within a healthy range, all of which are positive signs for your heart health. However, your resting blood pressure of 128 mmHg is considered elevated, meaning it's a bit higher than ideal, and your cholesterol at 210 mg/dL is slightly above the healthy target. While these values are not in the high-risk category, keeping your blood pressure and cholesterol levels in check through lifestyle choices like a balanced diet and regular physical activity can help support your long-term cardiovascular health. We recommend discussing these values with a healthcare professional to understand them better and determine if any steps are needed.

### Test case 4

In [9]:
# Test Case Edge: Single Medically Nonsensical Value
inputs_edge = {
    "Age":              45,
    "Sex":              "Male",
    "ChestPainType":    "ATA",
    "RestingBP":        0,         # zero BP = clinically dead (nonsensical)
    "Cholesterol":      210,
    "FastingBS":        100,
    "RestingECG":       "Normal",
    "MaxHR":            150,
    "ExerciseAngina":   "No",
    "Oldpeak":          0.8,
    "STSlope":          "Up"
}
inputs_edge["prediction"] = get_prediction(inputs_edge)

from IPython.display import display, Markdown
print("Test Case Edge — Single Nonsensical Value (RestingBP = 0)")
result_edge = call_gemini(SYSTEM_T1, USER_T1.format(**inputs_edge))
display(Markdown(result_edge))

Test Case Edge — Single Nonsensical Value (RestingBP = 0)


Your assessment suggests no heart disease detected, which is reassuring news. Looking at your results, your Resting ECG is normal and you did not experience any chest pain during exercise, which are positive indicators for your heart health. However, a couple of your values are on the borderline side. Your total cholesterol is 210 mg/dL, which is slightly above the ideal range and can be a factor in heart health over time. Your fasting blood sugar, at 100 mg/dL, is at the upper end of the healthy range, sometimes called pre-diabetes, and keeping it in check is important for long-term well-being. Additionally, while the overall assessment is reassuring, you reported a type of chest pain (ATA) that is sometimes associated with heart conditions, though it can also have other causes. Focusing on a balanced diet rich in fruits, vegetables, and whole grains, along with regular physical activity, can help manage both cholesterol and blood sugar levels effectively. Given these borderline values, it would be beneficial to discuss these results with your doctor to understand them further and explore any personalized recommendations they might have for you.

## Evaluation Summary



### Qualitative Evaluation

### 1. Edge-Case Handling

| Metric | Evaluation |
|---|---|
| Edge-Case Handling | The model did not flag the implausible RestingBP value of 0 mmHg as clinically impossible (TC4). Instead it proceeded to interpret the other values normally and produced a confident response without any caution about the RestingBP of 0 mmHg. This represents a notable limitation of the template in its current form. |

---

### 2. Instruction Following

| Sub-category | Evaluation |
|---|---|
| Personalisation | The template performs strongly on personalisation across all four test cases. TC1 referenced the exact blood pressure, cholesterol, fasting blood sugar, ST depression, and ST slope values. TC2 named the specific healthy values and explained why each one was reassuring. TC3 identified the borderline blood pressure and cholesterol individually. TC4 referenced the cholesterol, fasting blood sugar, and chest pain type specifically. No two responses were identical despite sharing the same input structure, confirming that the response is consistently driven by individual values rather than the prediction label alone. |
| Context | The model maintained the cardiovascular interpretation context consistently across all four test cases (TC1, TC2, TC3, TC4). No context drift was observed. Responses were always framed as patient-facing cardiovascular insights and the model never drifted into acting as a general chatbot or providing unrelated information. |
| Safety — No Diagnosis | The template follows this constraint consistently across all four test cases. TC1 uses "suggests the presence of heart disease", TC2 uses "suggests no heart disease", and TC3 uses "it suggests no heart disease was detected", all of which frame the result as a probabilistic assessment rather than a confirmed diagnosis. TC4 uses "suggests no heart disease detected" which is similarly cautious. The word "suggests" is consistently present across all responses, ensuring findings are always presented as possibilities rather than confirmed facts. |

### Quantitative Evaluation

#### 1. Flesch-Kincaid Readability Score



In [6]:
import nltk
import textstat

nltk.download('cmudict')

# T1 outputs
outputs_1 = {
    "TC1 — Heart Disease Detected": """Your assessment suggests the presence of heart disease, which is indicated by several key values. Your resting blood pressure of 140 mmHg and cholesterol level of 240 mg/dL are higher than recommended, indicating increased strain on your heart and arteries. Additionally, your fasting blood sugar of 160 mg/dL is elevated, which can impact your blood vessels over time. You also reported typical chest pain, especially during exercise, along with significant changes on your exercise ECG, specifically a 2.3 mm ST depression and a flat ST slope, which are strong indicators of reduced blood flow to your heart during physical activity. Taking steps to manage your blood pressure, cholesterol, and blood sugar, such as adopting a balanced diet and regular physical activity, can be beneficial. Given these results, it's important to consult with a medical professional to discuss these findings further and explore appropriate next steps for your heart health.""",

    "TC2 — No HD: All Healthy": """Your assessment suggests no heart disease, which is a very positive indication for your cardiovascular health. Your blood pressure of 110 mmHg is well within a healthy range, meaning your heart and blood vessels are not under undue strain. Similarly, your cholesterol level of 180 mg/dL and fasting blood sugar of 80 mg/dL are both healthy, which helps prevent blockages in your arteries and supports overall well-being. Additionally, your resting ECG is normal, and during exercise, you showed no signs of angina or concerning changes like ST depression, indicating your heart functions well under stress. Continuing to maintain these healthy habits through a balanced diet and regular physical activity will further support your excellent heart health.""",

    "TC3 — No HD: Borderline": """Based on your assessment, it suggests no heart disease was detected, which is generally reassuring. We observed that your resting ECG is normal, you don't experience chest pain indicating angina, and your blood sugar is within a healthy range, all of which are positive signs for your heart health. However, your resting blood pressure of 128 mmHg is considered elevated, meaning it's a bit higher than ideal, and your cholesterol at 210 mg/dL is slightly above the healthy target. While these values are not in the high-risk category, keeping your blood pressure and cholesterol levels in check through lifestyle choices like a balanced diet and regular physical activity can help support your long-term cardiovascular health. We recommend discussing these values with a healthcare professional to understand them better and determine if any steps are needed.""",

    "TC4 — Edge Case": """Your assessment suggests no heart disease detected, which is reassuring news. Looking at your results, your Resting ECG is normal and you did not experience any chest pain during exercise, which are positive indicators for your heart health. However, a couple of your values are on the borderline side. Your total cholesterol is 210 mg/dL, which is slightly above the ideal range and can be a factor in heart health over time. Your fasting blood sugar, at 100 mg/dL, is at the upper end of the healthy range, sometimes called pre-diabetes, and keeping it in check is important for long-term well-being. Additionally, while the overall assessment is reassuring, you reported a type of chest pain (ATA) that is sometimes associated with heart conditions, though it can also have other causes. Focusing on a balanced diet rich in fruits, vegetables, and whole grains, along with regular physical activity, can help manage both cholesterol and blood sugar levels effectively. Given these borderline values, it would be beneficial to discuss these results with your doctor to understand them further and explore any personalized recommendations they might have for you."""
}

results_readability_1 = evaluate_readability(outputs_1)

Flesch-Kincaid Readability Scores:
--------------------------------------------------
TC1 — Heart Disease Detected: 39.31
TC2 — No HD: All Healthy: 38.88
TC3 — No HD: Borderline: 37.18
TC4 — Edge Case: 42.69

Template Average: 39.52


[nltk_data] Error loading cmudict: <urlopen error [WinError 10054] An
[nltk_data]     existing connection was forcibly closed by the remote
[nltk_data]     host>


| Test Case | Score |
|---|---|
| TC1 — Heart Disease Detected | 39.31 |
| TC2 — No HD: All Healthy | 38.88 |
| TC3 — No HD: Borderline | 37.18 |
| TC4 — Edge Case | 42.69 |
| **Template Average** | **39.52** |

The template achieved an average Flesch-Kincaid readability score of
39.52. Since this template is designed for a first-time patient
encountering their clinical data for the very first time, the language
should be clear, accessible, and educational without being overly
detailed or technical. A score of 39.52 suggests the responses lean
towards complexity, which may make it harder for the patient to
quickly grasp the key message of their insight on a mobile screen.

#### 2. Domain Keyword Density




In [21]:
# T1 outputs
outputs_1 = {
    "TC1 — Heart Disease Detected": """Your assessment suggests the presence of heart disease, which is indicated by several key values. Your resting blood pressure of 140 mmHg and cholesterol level of 240 mg/dL are higher than recommended, indicating increased strain on your heart and arteries. Additionally, your fasting blood sugar of 160 mg/dL is elevated, which can impact your blood vessels over time. You also reported typical chest pain, especially during exercise, along with significant changes on your exercise ECG, specifically a 2.3 mm ST depression and a flat ST slope, which are strong indicators of reduced blood flow to your heart during physical activity. Taking steps to manage your blood pressure, cholesterol, and blood sugar, such as adopting a balanced diet and regular physical activity, can be beneficial. Given these results, it's important to consult with a medical professional to discuss these findings further and explore appropriate next steps for your heart health.""",

    "TC2 — No HD: All Healthy": """Your assessment suggests no heart disease, which is a very positive indication for your cardiovascular health. Your blood pressure of 110 mmHg is well within a healthy range, meaning your heart and blood vessels are not under undue strain. Similarly, your cholesterol level of 180 mg/dL and fasting blood sugar of 80 mg/dL are both healthy, which helps prevent blockages in your arteries and supports overall well-being. Additionally, your resting ECG is normal, and during exercise, you showed no signs of angina or concerning changes like ST depression, indicating your heart functions well under stress. Continuing to maintain these healthy habits through a balanced diet and regular physical activity will further support your excellent heart health.""",

    "TC3 — No HD: Borderline": """Based on your assessment, it suggests no heart disease was detected, which is generally reassuring. We observed that your resting ECG is normal, you don't experience chest pain indicating angina, and your blood sugar is within a healthy range, all of which are positive signs for your heart health. However, your resting blood pressure of 128 mmHg is considered elevated, meaning it's a bit higher than ideal, and your cholesterol at 210 mg/dL is slightly above the healthy target. While these values are not in the high-risk category, keeping your blood pressure and cholesterol levels in check through lifestyle choices like a balanced diet and regular physical activity can help support your long-term cardiovascular health. We recommend discussing these values with a healthcare professional to understand them better and determine if any steps are needed.""",

    "TC4 — Edge Case": """Your assessment suggests no heart disease detected, which is reassuring news. Looking at your results, your Resting ECG is normal and you did not experience any chest pain during exercise, which are positive indicators for your heart health. However, a couple of your values are on the borderline side. Your total cholesterol is 210 mg/dL, which is slightly above the ideal range and can be a factor in heart health over time. Your fasting blood sugar, at 100 mg/dL, is at the upper end of the healthy range, sometimes called pre-diabetes, and keeping it in check is important for long-term well-being. Additionally, while the overall assessment is reassuring, you reported a type of chest pain (ATA) that is sometimes associated with heart conditions, though it can also have other causes. Focusing on a balanced diet rich in fruits, vegetables, and whole grains, along with regular physical activity, can help manage both cholesterol and blood sugar levels effectively. Given these borderline values, it would be beneficial to discuss these results with your doctor to understand them further and explore any personalized recommendations they might have for you."""
}

# Evaluate domain keyword density using reference list
results_1 = evaluate_keyword_density(outputs_1)

Domain Keyword Density:
--------------------------------------------------
TC1 — Heart Disease Detected:
  Total Words       : 149
  Domain Keywords   : 15
  Density           : 10.07%

TC2 — No HD: All Healthy:
  Total Words       : 116
  Domain Keywords   : 8
  Density           : 6.9%

TC3 — No HD: Borderline:
  Total Words       : 135
  Domain Keywords   : 7
  Density           : 5.19%

TC4 — Edge Case:
  Total Words       : 186
  Domain Keywords   : 9
  Density           : 4.84%

Template Average Density: 6.75%


| Test Case | Domain Keywords | Total Words | Density |
|---|---|---|---|
| TC1 — Heart Disease Detected | 15 | 149 | 10.07% |
| TC2 — No HD: All Healthy | 8 | 116 | 6.90% |
| TC3 — No HD: Borderline | 7 | 135 | 5.19% |
| TC4 — Edge Case | 9 | 186 | 4.84% |
| **Template Average** | | | **6.75%** |

The template achieves a low domain keyword density across all test
cases, which is consistent with the intended user profile of a
first-time patient with no medical background. A low density
indicates that the responses avoid overloading the patient with
clinical terminology, prioritising accessibility and clarity over
technical depth. This is appropriate for a patient encountering
their cardiovascular data for the very first time on a mobile screen.

#### 3. Medical Accuracy

| # | Test Case | Clinical Claim | Correct |
|---|---|---|---|
| 1 | TC1 | Blood pressure of 140 mmHg is higher than recommended | Yes |
| 2 | TC1 | Cholesterol of 240 mg/dL is higher than recommended | Yes |
| 3 | TC1 | Fasting blood sugar of 160 mg/dL is elevated | No |
| 4 | TC2 | Blood pressure of 110 mmHg is within a healthy range | Yes |
| 5 | TC2 | Cholesterol of 180 mg/dL is healthy | Yes |
| 6 | TC2 | Fasting blood sugar of 80 mg/dL is healthy | Yes |
| 7 | TC3 | Blood pressure of 128 mmHg is considered elevated | Yes |
| 8 | TC3 | Cholesterol of 210 mg/dL is slightly above the healthy target | Yes |
| 9 | TC3 |  blood sugar is within a healthy range | No |
| 10 | TC4 | Cholesterol of 210 mg/dL is slightly above the ideal range | Yes |
| 11 | TC4 | Fasting blood sugar of 100 mg/dL is at the upper end of the healthy range | No |


---

| Test Case | Correct Claims | Total Claims | Accuracy Rate |
|---|---|---|---|
| TC1 | 2 | 3 | 66.67% |
| TC2 | 3 | 3 | 100% |
| TC3 | 2 | 3 | 66.67% |
| TC4 | 1 | 2 | 50% |
| **Template Average** | | | **70.84%** |

#### 4. Passive Voice Rate



In [22]:
# T1 outputs
outputs_1 = {
    "TC1 — Heart Disease Detected": """Your assessment suggests the presence of heart disease, which is indicated by several key values. Your resting blood pressure of 140 mmHg and cholesterol level of 240 mg/dL are higher than recommended, indicating increased strain on your heart and arteries. Additionally, your fasting blood sugar of 160 mg/dL is elevated, which can impact your blood vessels over time. You also reported typical chest pain, especially during exercise, along with significant changes on your exercise ECG, specifically a 2.3 mm ST depression and a flat ST slope, which are strong indicators of reduced blood flow to your heart during physical activity. Taking steps to manage your blood pressure, cholesterol, and blood sugar, such as adopting a balanced diet and regular physical activity, can be beneficial. Given these results, it's important to consult with a medical professional to discuss these findings further and explore appropriate next steps for your heart health.""",

    "TC2 — No HD: All Healthy": """Your assessment suggests no heart disease, which is a very positive indication for your cardiovascular health. Your blood pressure of 110 mmHg is well within a healthy range, meaning your heart and blood vessels are not under undue strain. Similarly, your cholesterol level of 180 mg/dL and fasting blood sugar of 80 mg/dL are both healthy, which helps prevent blockages in your arteries and supports overall well-being. Additionally, your resting ECG is normal, and during exercise, you showed no signs of angina or concerning changes like ST depression, indicating your heart functions well under stress. Continuing to maintain these healthy habits through a balanced diet and regular physical activity will further support your excellent heart health.""",

    "TC3 — No HD: Borderline": """Based on your assessment, it suggests no heart disease was detected, which is generally reassuring. We observed that your resting ECG is normal, you don't experience chest pain indicating angina, and your blood sugar is within a healthy range, all of which are positive signs for your heart health. However, your resting blood pressure of 128 mmHg is considered elevated, meaning it's a bit higher than ideal, and your cholesterol at 210 mg/dL is slightly above the healthy target. While these values are not in the high-risk category, keeping your blood pressure and cholesterol levels in check through lifestyle choices like a balanced diet and regular physical activity can help support your long-term cardiovascular health. We recommend discussing these values with a healthcare professional to understand them better and determine if any steps are needed.""",

    "TC4 — Edge Case": """Your assessment suggests no heart disease detected, which is reassuring news. Looking at your results, your Resting ECG is normal and you did not experience any chest pain during exercise, which are positive indicators for your heart health. However, a couple of your values are on the borderline side. Your total cholesterol is 210 mg/dL, which is slightly above the ideal range and can be a factor in heart health over time. Your fasting blood sugar, at 100 mg/dL, is at the upper end of the healthy range, sometimes called pre-diabetes, and keeping it in check is important for long-term well-being. Additionally, while the overall assessment is reassuring, you reported a type of chest pain (ATA) that is sometimes associated with heart conditions, though it can also have other causes. Focusing on a balanced diet rich in fruits, vegetables, and whole grains, along with regular physical activity, can help manage both cholesterol and blood sugar levels effectively. Given these borderline values, it would be beneficial to discuss these results with your doctor to understand them further and explore any personalized recommendations they might have for you."""
}
results_passive_1 = evaluate_passive_voice(outputs_1)

Passive Voice Rate:
--------------------------------------------------
TC1 — Heart Disease Detected:
  Total Sentences  : 6
  Passive Sentences: 2
  Passive Rate     : 33.33%

TC2 — No HD: All Healthy:
  Total Sentences  : 5
  Passive Sentences: 0
  Passive Rate     : 0.0%

TC3 — No HD: Borderline:
  Total Sentences  : 5
  Passive Sentences: 3
  Passive Rate     : 60.0%

TC4 — Edge Case:
  Total Sentences  : 8
  Passive Sentences: 1
  Passive Rate     : 12.5%

Template Average Passive Voice Rate: 26.46%


| Test Case | Passive Sentences | Total Sentences | Passive Rate |
|---|---|---|---|
| TC1 — Heart Disease Detected | 2 | 6 | 33.33% |
| TC2 — No HD: All Healthy | 0 | 5 | 0.00% |
| TC3 — No HD: Borderline | 3 | 5 | 60.00% |
| TC4 — Edge Case | 1 | 8 | 12.50% |
| **Template Average** | | | **26.46%** |

The template shows inconsistent passive voice usage across test cases.
TC2 performs well with a passive rate of 0%, indicating a fully
active and patient-centred response. TC3 however shows a notably
high passive rate of 60%, which makes the response feel more like
a clinical report than a direct patient-facing insight. TC1 and TC4
fall in between. The overall template average of 26.46% suggests
there is room for improvement in ensuring the responses consistently
address the patient directly using active voice.

## Assumptions and Limitations

### Assumptions


**1. First-Time Patient Profile**
The patient is assumed to be a first-time user with no medical background
and no prior frame of reference for their clinical values. The prompt
design, tone, and output format were all calibrated around this assumption.
If the system were deployed for returning patients or those with medical
knowledge, the template would require significant adaptation.

**2. Human-Readable Feature Values**
The clinical feature values passed to the prompt are assumed to be in
human-readable form before one-hot encoding.

**3. Hospital Mobile Application Deployment**
The system is assumed to be deployed in a hospital mobile application
where patients have already completed data entry. The prompt is designed
for a single patient assessment at a time and does not account for
batch processing or multi-patient scenarios.


**4. Internal Knowledge Influences the Final Response**
It is assumed that the knowledge generated internally in Stage 1
directly and meaningfully influences the final insight produced in
Stage 2. Without this assumption the two-stage structure of Generated
Knowledge Prompting would have no practical benefit over a direct
single-stage response.


**5. Single Prompt Sufficiency**
It is assumed that a single prompt request is sufficient to complete
both stages of the Generated Knowledge Prompting process. This assumes
the model can internally manage the knowledge generation and insight
production steps without requiring multiple separate interactions.

---

### Limitations

**1. No External Knowledge Source**
The template relies entirely on the model's internal pre-trained
knowledge to generate clinical context in Stage 1. Unlike retrieval
augmented approaches, no external medical database, clinical
guideline, or verified knowledge source is consulted during the
knowledge generation stage. This means the accuracy and completeness
of the generated knowledge depends solely on what the model has
learned during pre-training.


**2. Knowledge Generation Cannot Be Controlled**
The prompt instructs the model to generate clinical knowledge
internally but has no control over what specific knowledge is
generated, how much is generated, or how deeply each value is
analysed. The depth and relevance of the knowledge generated in
Stage 1 varies depending on the model's internal priorities rather
than a structured or predefined clinical framework.

**3. Knowledge Staleness**
The clinical knowledge generated internally by the model is based
on its pre-training data which has a fixed knowledge cutoff. As
cardiovascular guidelines and clinical standards evolve over time,
the model's internal knowledge may become outdated without any
mechanism to update or correct it.

# Chain-of-Thought Prompting

### Template ID
2

---

### Approach Definition

Chain-of-thought (CoT) prompting is a prompt engineering technique in which a language model is instructed to produce a series of intermediate reasoning steps before arriving at a final answer. Rather than mapping an input directly to an output in a single pass, the model externalizes its thinking process, each step builds on the previous one, and the final answer is explicitly grounded in the visible reasoning chain. IBM describes CoT as a technique that simulates human-like reasoning by breaking down elaborate problems into manageable, intermediate steps that sequentially lead to a conclusive answer (IBM, 2024). The technique is activated by including a structured reasoning directive in the prompt, most commonly the phrase "think step by step," which signals to the model that intermediate steps are required as part of the response rather than just a final conclusion.


> IBM. (2024). *What is chain of thought (CoT) prompting?* IBM Think.
> __https://www.ibm.com/think/topics/chain-of-thoughts__

---

### Relation to Medical Field

The structure of chain-of-thought prompting mirrors the way clinicians think when making a diagnosis. A doctor assessing a patient for cardiovascular disease does not jump straight from test results to a treatment decision. Patel and Groen (1986) studied how cardiologists reason through complex cases and found that accurate diagnoses were reached by moving in one direction only: from clinical findings, step by step, toward a conclusion. Each step followed from the one before it, building on the available evidence rather than starting from an assumed answer. This is exactly how chain-of-thought prompting works: the model moves forward through a series of reasoning steps, each grounded in the previous one, without going back to revise earlier conclusions, making CoT a naturally suitable technique for clinical decision-making tasks.

> Patel, V. L., & Groen, G. J. (1986). Knowledge based solution strategies in medical reasoning. *Cognitive Science*, *10*(1), 91–116.
> https://doi.org/10.1207/s15516709cog1001_4
---

### Intended Use Case

**1- Deployment Environment**

The system is planned to be deployed as an interpretation tool within a hospital
mobile application. Patients access it from their personal smartphones
after obtaining their clinical data, either from a recent cardiovascular
test or by completing one at the hospital. The patient enters their
clinical values into the tool and receives a personalized interpretation
directly on the same screen.

**2- The User**

The primary user of this template is a detail-oriented patient who approaches their health result with curiosity and a desire to understand rather than simply act. This patient is likely to be someone who has prior experience with cardiovascular health (either through a personal history of monitoring their health closely, a family member with heart disease, or a general habit of researching their medical information). They are comfortable reading a longer, structured response and will invest the time to follow a step-by-step explanation from beginning to conclusion.
Research in patient health literacy confirms that a significant proportion of patients actively prefer detailed explanations of their results, as understanding the reasoning behind a recommendation increases their confidence in acting on it and their trust in the system delivering it (Zolnierek and Dimatteo, 2009).
> Zolnierek, K.B.H. and DiMatteo, M.R. (2009). *Physician communication
> and patient adherence to treatment: A meta-analysis.* Medical Care,
> 47(8), 826–834.
> https://doi.org/10.1097/MLR.0b013e31819a5acc

***Medical background:*** While no formal medical background is assumed, this patient is health-literate. They are capable of understanding concepts such as blood pressure ranges, cholesterol categories, and the general relationship between lifestyle factors and cardiovascular risk. The interpretation may therefore use slightly more specific language than the zero-shot template, as long as each term is explained in context. (Is it okay to compare here?)

***Emotional state:*** This patient is engaged rather than acutely anxious. They are seeking information and understanding, which means a longer, more structured response will not increase their distress. However, the tone must still remain calm and professional. The visibility of the reasoning steps must never make the output feel like a clinical verdict.

**3- Inputs**

3.1 The patient heart disease classification result

3.2 The patient clinical feature values


**4- Output**

A structured response consisting of a brief per-feature assessment followed by a synthesised final interpretation paragraph. The reasoning steps walk through the patient's most significant clinical values one by one, identifying what each means and how it contributes to the overall picture. The paragraph then synthesises these steps into a coherent, actionable recommendation. The visible reasoning is written in plain language throughout so that the patient can follow each step without a medical background. Scrolling is acceptable here because the patient prefers this level of details.

---

### Design Rationale

**Advantage 1 — Improved accuracy on complex reasoning tasks:**
Chain-of-thought improves model performance on complex tasks by breaking
them down into smaller, manageable steps, with each intermediate step
acting as a checkpoint where errors can be caught before they carry
forward into the final output (DataCamp, 2024). In a heart disease
interpretation system, this matters because an error made early (such as
misreading a borderline cholesterol value or misclassifying a resting
ECG result) will corrupt every reasoning step that follows it. By
making each step visible and sequential, chain-of-thought allows such
errors to be identified at the point they occur rather than arriving
silently embedded in a final recommendation.

**Advantage 2 — Transparency and explainability:** By generating visible
intermediate reasoning steps, chain-of-thought makes the decision-making
process auditable and easier to follow for non-expert users (IBM, 2024).
In a heart disease interpretation system where the intended users have no
medical background, this is a functional requirement. A patient who
receives only a label or a conclusion cannot understand what it
means for them personally, identify which of their values drove the
outcome, or communicate the basis of the interpretation to a doctor. By
walking through each feature in plain, step-by-step reasoning before
reaching a conclusion, chain-of-thought produces an output that a
non-specialist can follow, question, and act on, rather than simply
accept or dismiss.

**Advantage 3 — Multistep reasoning capability:** Chain-of-thought
handles tasks that require multistep reasoning by systematically working
through each component of a problem before reaching a conclusion, leading
to more reliable outputs (IBM, 2024). In cardiovascular test,
reliability is critical where a premature conclusion or a skipped step can
result in an interpretation that does not reflect the patient's actual
condition. By structuring the output as a sequence of dependent reasoning
steps, chain-of-thought ensures the model cannot arrive at a conclusion
before all components of the problem have been addressed, making the
final output consistent and reproducible regardless of how the input
values are distributed.

**Advantage 4 — Attention to detail:** The step-by-step nature of
chain-of-thought encourages detailed breakdowns of complex inputs,
ensuring each component is examined and accounted for before a conclusion
is reached (IBM, 2024). In cardiovascular test, where no
single factor is decisive on its own, this level of detail is essential.
The ACC/AHA Pooled Cohort Equations calculate cardiovascular risk from
multiple variables whose combined effect is non-additive, meaning the
contribution of any one feature depends on the values of the others
(Goff et al., 2014). Chain-of-thought ensures that no feature is skipped
or treated as insignificant, making it the most detail-preserving
technique available for this task.


> DataCamp. (2024). *Chain-of-thought prompting*.
> https://www.datacamp.com/tutorial/chain-of-thought-prompting

> IBM. (2024). *Chain of thoughts*. IBM Think.
> https://www.ibm.com/think/topics/chain-of-thoughts

> Goff, D. C., Lloyd-Jones, D. M., Bennett, G., Coady, S., D'Agostino, R. B.,  Gibbons, R., Greenland, P., Lackland, D. T., Levy, D., O'Donnell, C. J.,  Robinson, J. G., Schwartz, J. S., Shero, S. T., Smith, S. C., Sorlie, P., Stone, N. J., & Wilson, P. W. F. (2014). 2013 ACC/AHA guideline on the assessment of cardiovascular risk: A report of the American College of Cardiology/American Heart Association Task Force on Practice Guidelines. *Journal of the American College of Cardiology*, *63*(25), 2935–2959.
> https://doi.org/10.1016/j.jacc.2013.11.005

---

### Prompt Engineering Techniques Applied

#### Role Assignment

*You are a cardiovascular health analyst deployed in a hospital mobile application.*

The role assigned to the model is cardiovascular health analyst. This was chosen deliberately over alternatives like "health advisor" or "health educator." An advisor gives recommendations. An educator builds knowledge in someone who lacks it. An analyst examines evidence and makes reasoning visible step by step, which is exactly what this template requires for a detail-oriented patient who wants to follow the thinking behind their result, not just receive a conclusion.

#### Role Description

*Your role is to help health-literate patients understand the results of their cardiovascular test by examining each clinical value in turn, making your reasoning visible, and then delivering a final personalised advisory.*

This sentence defines the three-stage job the model must perform: **(1)** examine each value individually, **(2)** make the reasoning visible at each step, and **(3)** deliver a final overview after all values have been analyzed. The phrase **"making your reasoning visible"** is what activates chain-of-thought behaviour. It tells the model that intermediate reasoning must appear in the output and must not be hidden or summarized internally.

#### User Context Priming
*The patient reading this response is health-literate and engaged. They are not a medical professional, but they are comfortable with concepts such as blood pressure categories, cholesterol levels, and the relationship between lifestyle habits and cardiovascular risk. They expect each of their test values to be explained individually and connected to the overall picture before a recommendation is made.*

This paragraph calibrates the vocabulary level the model should write at. It tells the model two things simultaneously: do not over-simplify (this patient knows what cholesterol and blood pressure mean), and do not assume clinical expertise (they cannot interpret an Oldpeak reading or ST slope without explanation).

#### Tone Specification

*Your tone must be calm, informative, and respectful. Do not be alarming, but do not soften findings to the point of obscuring their significance.*

This template prioritizes honest clarity over emotional reassurance. The intended users want a transparent analysis, where over-softening findings would frustrate them and defeat the purpose of visible reasoning.

#### Safety Constraint

*Present the overall finding naturally and as a probability.*

The system instruction requires the model to frame all clinical findings as possibilities rather than certainties. In a cardiovascular interpretation system, the model is not a licensed clinician and cannot account for the patient's full medical history. Epistemic hedging is therefore a fundamental safety requirement, not a stylistic preference.


#### Conditional Response Logic

*If the prediction indicates heart disease is detected: identify the specific clinical values that are outside healthy ranges and generate a personalised interpretation tied directly to those values. If the prediction indicates no heart disease is detected and all values are within healthy ranges: reassure the patient, reinforce the specific positive findings that support this outcome, and provide a practical advice. If one or more values are borderline — meaning they are technically within the healthy range but approaching concerning thresholds — provide a targeted interpretation for each borderline value, grounded in its specific value and what it may indicate clinically with a practical advice for how to address it.*

The interpretation follows three-branch logic based on the combination of the prediction label and the actual feature values. This is important because two patients can share the same prediction label but have completely different clinical profiles. A patient classified as no heart disease with all healthy values needs a different response from a patient classified as no heart disease but with a borderline blood pressure reading. The three branches ensure the interpretation is genuinely personalised rather than driven by the label alone.

#### Output Length Constraint

*The overview paragraph must be exactly 2 to 3 sentences.*

The system instruction specifies that the interpretation paragraph must be exactly 2 to 3 sentences. Google's prompting strategies documentation recommends being explicit about output constraints including length, noting that such constraints help the model produce concise and predictable responses and avoid unnecessary elaboration (Google, 2024). The 2 to 3 sentence limit was chosen because the interpretation is displayed first on the mobile screen and must be immediately readable without scrolling. This constraint is sufficient because the detailed reasoning is already present in the per-value tables displayed below the interpretation, the patient is not losing information, they are receiving it in two layers. In a complete system implementation, those tables would be exportable as a downloadable PDF that the patient can save, print, or bring to their next clinical appointment for a more thorough discussion with their doctor. The interpretation therefore only needs to deliver the overall finding and the most critical actionable points, leaving the full reasoning accessible to those who want to engage with it further.
> Google. (2024). *Introduction to prompting.* Google AI for Developers.
> https://ai.google.dev/gemini-api/docs/prompting-intro

---

### Prompt Structure

In [26]:
SYSTEM_T2 = """You are a cardiovascular health analyst deployed in a hospital \
mobile application. Your role is to help health-literate patients understand \
the results of their cardiovascular test by examining each clinical value in \
turn, making your reasoning visible, and then delivering a final personalised \
overview.

The patient reading this response is health-literate and engaged. They are \
not a medical professional, but they are comfortable with concepts such as \
blood pressure categories, cholesterol levels, and the relationship between \
lifestyle habits and cardiovascular risk. They expect each of their test \
values to be explained individually and connected to the overall picture \
before a recommendation is made.

Follow these rules at all times:
1. Always show your reasoning steps visibly in the output. Do NOT hide or \
   summarise the steps internally. Present your analysis in a clear, logical \
   flow — do not use any step labels, headers, or numbering such as \
   "Step 1", "Step 2", "Part 1", "Analysis", or any similar \
   structural label anywhere in your response.

2. Begin your response by analysing each cardiovascular test value. For each \
   value, present your findings in its own individual two-column table with \
   the following rows:
   | Test Value          | [value name and number] |
   | What It Measures    | [one sentence]          |
   | Assessment          | [one sentence]          |
   | What It May Suggest | [one sentence]          |

   Contextual values such as Age and Sex should have their \
   own small table with only the Test Value and Assessment rows filled — \
   leave What It Measures and What It May Suggest empty for those rows. \
   Reserve the full four-row table for clinically actionable values only — \
   those that can directly inform lifestyle recommendations or signal \
   cardiovascular risk.

   Contextual values such as Age and Sex should follow the same format but \
   with only the Assessment bullet point filled — omit What It Measures and \
   What It May Suggest for those. Reserve the full three-bullet format for \
   clinically actionable values only — those that can directly inform \
   lifestyle recommendations or signal cardiovascular risk.

3. After completing the table for every test value, insert this exact line \
   on its own with nothing before or after it on that line:
   ---Overview---
   Then, immediately after that line, write a single coherent \
   overview paragraph. The content of this paragraph must be determined by \
   the prediction and the individual clinical values, as follows:

   If the prediction indicates heart disease is detected: identify the \
   specific clinical values that are outside healthy ranges and generate \
   a personalised interpretation tied directly to those values. \
   If the prediction indicates no heart disease is detected and all values \
   are within healthy ranges: reassure the patient, reinforce the \
   specific positive findings that support this outcome, and provide a practical advice. \
   If one or more values are borderline — meaning they are technically \
   within the healthy range but approaching concerning thresholds — provide \
   a targeted interpretation for each borderline value, grounded in its specific value and what it may \
   indicate clinically with a practical advice for how to address it.

4. The overview paragraph must be exactly 2 to 3 sentences. Acknowledge the \
   overall finding in the first sentence. Use the next 1 to 2 sentences to \
   address the most significant findings with specific recommendations \
   tied directly to the named clinical values.

5. Never state a diagnosis and never reference the classification model \
   directly in your output. Do not use phrases like "the model predicts," \
   "the classification result," or "the prediction." Do not use phrases like \
   "heart disease is not currently detected" or "heart disease is detected" \
   or any variation that directly states a diagnostic conclusion. Present the \
   overall finding naturally and as a probability.

6. Write in plain, clear language throughout. You may use health-literacy-level \
   terms but always explain what each term \
   means immediately after using it in one plain sentence.

7. Your tone must be calm, informative, and respectful throughout — as though \
   you are a knowledgeable analyst walking the patient through their test \
   results with patience and clarity. Do not be alarming, but do not soften \
   findings to the point of obscuring their significance.

8. Every recommendation or warning in the overview paragraph must name the \
   specific test value it addresses and explain directly why that value makes \
   this recommendation or warning necessary. Generic advice with no traceable \
   link to a specific finding is not acceptable.
"""


USER_T2 = """The following result was produced by a heart disease classification \
model trained on cardiovascular test data. Using the structure defined in your \
instructions, analyse each test value and deliver the final overview.

Prediction     : {prediction}

Cardiovascular test results:
  Age                    : {Age} years
  Sex                    : {Sex}
  Chest Pain Type        : {ChestPainType}
  Resting Blood Pressure : {RestingBP} mmHg
  Cholesterol            : {Cholesterol} mg/dL
  Fasting Blood Sugar    : {FastingBS} mg/dL
  Resting ECG            : {RestingECG}
  Max Heart Rate         : {MaxHR} bpm
  Exercise Angina        : {ExerciseAngina}
  Oldpeak                : {Oldpeak}
  ST Slope               : {STSlope}"""

def call_T2(inputs):
    inputs["prediction"] = get_prediction(inputs)

    response = call_gemini(
        SYSTEM_T2,
        USER_T2.format(**inputs)
    )

    from IPython.display import Markdown, display
    import re
    parts = response.split("---Overview---")
    step1_tables   = parts[0].strip()
    tables = re.split(r'(?m)^(?=\|.*Test Value)', step1_tables)
    tables = [t.strip() for t in tables if t.strip()]
    step2_overview = parts[1].strip() if len(parts) > 1 else ""

    display(Markdown(step2_overview))
    display(Markdown("\nThe following represents more details regarding each of your cardiovascular test values:\n"))

    for table in tables:
       display(Markdown(table))

   # Return both parts for analysis
    return step2_overview, step1_tables

### Input/Output Examples

In [36]:
# Test Case 1: Heart Disease Detected
inputs = {
    "prediction":       "",
    "Age":              55,
    "Sex":              "Male",
    "ChestPainType":  "ATA",
    "RestingBP":       140,
    "Cholesterol":      240,
    "FastingBS":       160,
    "RestingECG":      "Normal",
    "MaxHR":           150,
    "ExerciseAngina":  "Yes",
    "Oldpeak":          2.3,
    "STSlope":         "Flat"
}

tc1_advisory, tc1_tables = call_T2(inputs)

Your test results indicate several factors that collectively suggest an increased likelihood of cardiovascular concerns. Given your elevated **Resting Blood Pressure** of 140 mmHg, high **Cholesterol** at 240 mg/dL, and **Fasting Blood Sugar** of 160 mg/dL, focusing on lifestyle adjustments and medical guidance to manage these levels is crucial, especially alongside the reported **Exercise Angina**, significant **Oldpeak** of 2.3, and **Flat ST Slope** which suggest potential reduced blood flow to your heart during physical activity.


The following represents more details regarding each of your cardiovascular test values:


| Test Value | Age: 55 years |
|---|---|
| Assessment | This indicates your age at the time of the test. |

| Test Value | Sex: Male |
|---|---|
| Assessment | This indicates your biological sex. |

| Test Value | Chest Pain Type: ATA |
|---|---|
| What It Measures | This describes the type of chest discomfort experienced. |
| Assessment | Atypical angina refers to chest pain that has some characteristics of angina but doesn't fit the typical pattern. |
| What It May Suggest | While not definitively heart-related, atypical angina can sometimes be associated with cardiovascular issues. |

...

| Test Value | Resting ECG: Normal |
|---|---|
| What It Measures | An electrocardiogram (ECG) records the electrical activity of your heart at rest. |
| Assessment | Your resting ECG result is noted as normal. |
| What It May Suggest | A normal resting ECG suggests that there are no obvious electrical abnormalities in your heart rhythm or structure detectable at rest. |

| Test Value | Max Heart Rate: 150 bpm |
|---|---|
| What It Measures | This is the highest heart rate achieved during exercise or exertion. |
| Assessment | Your maximum heart rate during the test was 150 beats per minute. |
| What It May Suggest | This value is often assessed in context with age and exercise capacity; for a 55-year-old, this may be considered a reasonable response to exertion. |

| Test Value | Exercise Angina: Yes |
|---|---|
| What It Measures | This indicates whether you experienced chest pain or discomfort during physical exertion. |
| Assessment | You reported experiencing angina during exercise. |
| What It May Suggest | Chest pain during exercise is a common symptom of coronary artery disease, where the heart muscle isn't getting enough blood flow during increased demand. |

| Test Value | Oldpeak: 2.3 |
|---|---|
| What It Measures | This value relates to ST depression measured during an exercise ECG, indicating potential reduced blood flow to the heart. |
| Assessment | Your Oldpeak value is 2.3, which represents a significant ST depression during exercise. |
| What It May Suggest | Significant ST depression during exercise strongly suggests myocardial ischemia, meaning your heart muscle is not receiving enough blood. |

| Test Value | ST Slope: Flat |
|---|---|
| What It Measures | This describes the slope of the ST segment of the ECG during exercise, which can indicate how well blood flows to the heart. |
| Assessment | Your ST segment slope during exercise was flat. |
| What It May Suggest | A flat ST segment slope during exercise is often associated with myocardial ischemia, indicating potential blockages in the coronary arteries. |

In [53]:
# Test Case 2: No Heart Disease Detected - All Healthy Values
inputs = {
    "prediction":       "",
    "Age":              35,
    "Sex":              "Female",
    "ChestPainType":  "NAP",
    "RestingBP":       110,
    "Cholesterol":      180,
    "FastingBS":       80,
    "RestingECG":      "Normal",
    "MaxHR":           175,
    "ExerciseAngina":  "No",
    "Oldpeak":          0.0,
    "STSlope":         "Up"
}

tc2_advisory, tc2_tables = call_T2(inputs)

It is very encouraging to see that your cardiovascular test results show a low probability of heart disease, with all your individual values indicating excellent health. Your Resting Blood Pressure of 110 mmHg, Cholesterol of 180 mg/dL, and Fasting Blood Sugar of 80 mg/dL are all optimal, demonstrating a strong foundation for cardiovascular wellness. Continue to maintain these healthy habits, such as regular physical activity and a balanced diet, to support your heart health in the long term.


The following represents more details regarding each of your cardiovascular test values:


| Test Value | Age: 35 years |
| :------------------ | :------------------------------------ |
| Assessment | This indicates your current chronological age. |

| Test Value | Sex: Female |
| :------------------ | :------------------------------------ |
| Assessment | This indicates your biological sex. |

| Test Value | Chest Pain Type: NAP |
| :------------------ | :---------------------------------------------------------------------------------------------------------------------- |
| What It Measures | This describes the nature of any chest discomfort experienced. |
| Assessment | Your chest pain is classified as Non-Anginal Pain (NAP), which means it is typically not related to a heart condition. |
| What It May Suggest | This finding is reassuring as it does not suggest chest pain originating from reduced blood flow to the heart. |

...

| Test Value | Resting ECG: Normal |
| :------------------ | :---------------------------------------------------------------------------------------------------------------------- |
| What It Measures | This test records the electrical activity of your heart to detect any abnormalities. |
| Assessment | Your Resting ECG shows normal electrical patterns of your heart. |
| What It May Suggest | A normal ECG suggests that your heart's electrical system is functioning correctly without significant signs of strain or damage at rest. |

| Test Value | Max Heart Rate: 175 bpm |
| :------------------ | :---------------------------------------------------------------------------------------------------------------------- |
| What It Measures | This is the highest heart rate achieved during exercise or physical exertion. |
| Assessment | A maximum heart rate of 175 bpm during exertion is considered a healthy and appropriate response for your age. |
| What It May Suggest | This suggests good cardiovascular fitness and your heart's ability to respond to physical demands. |

| Test Value | Exercise Angina: No |
| :------------------ | :---------------------------------------------------------------------------------------------------------------------- |
| What It Measures | This indicates whether you experienced chest pain during physical exertion or exercise. |
| Assessment | You did not experience any chest pain during exercise. |
| What It May Suggest | The absence of exercise angina is a very positive sign, indicating that your heart is likely receiving sufficient blood flow during physical activity. |

| Test Value | Oldpeak: 0.0 |
| :------------------ | :---------------------------------------------------------------------------------------------------------------------- |
| What It Measures | This value reflects the amount of ST depression measured during exercise relative to your resting electrocardiogram. |
| Assessment | An Oldpeak value of 0.0 indicates no significant ST depression during exercise. |
| What It May Suggest | This suggests that your heart muscle is receiving adequate blood supply even under the stress of exercise. |

| Test Value | ST Slope: Up |
| :------------------ | :---------------------------------------------------------------------------------------------------------------------- |
| What It Measures | This describes the direction of the ST segment change on an electrocardiogram during exercise. |
| Assessment | An 'Up' (upsloping) ST segment is considered a normal and reassuring finding during exercise. |
| What It May Suggest | This indicates a healthy response of your heart's electrical activity during physical stress. |

In [54]:
# Test Case 3: No Heart Disease Detected - Borderline Values
inputs = {
    "prediction":       "",
    "Age":              45,
    "Sex":              "Male",
    "ChestPainType":  "NAP",
    "RestingBP":       128,
    "Cholesterol":      210,
    "FastingBS":       100,
    "RestingECG":      "Normal",
    "MaxHR":           160,
    "ExerciseAngina":  "No",
    "Oldpeak":          0.8,
    "STSlope":         "Up"
}

tc3_advisory, tc3_tables = call_T2(inputs)

Your results indicate a low probability of significant heart disease at this time, supported by a normal resting ECG, no exercise-induced chest pain, and a healthy ST slope during exertion. Nevertheless, it's worth noting your Resting Blood Pressure of 128 mmHg, Cholesterol at 210 mg/dL, and Fasting Blood Sugar of 100 mg/dL are at the higher end of optimal ranges, along with a mild Oldpeak of 0.8, suggesting proactive lifestyle adjustments like dietary changes and regular exercise could help keep these values in a healthier zone.


The following represents more details regarding each of your cardiovascular test values:


| Test Value | 45 years |
|---|---|
| Assessment | Your age of 45 places you in a demographic where cardiovascular health becomes an important focus for preventive care. |

| Test Value | Male |
|---|---|
| Assessment | Being male is a non-modifiable factor often considered in cardiovascular risk assessment. |

| Test Value | Chest Pain Type: NAP |
|---|---|
| What It Measures | This indicates the type of discomfort you experienced in your chest, with "NAP" meaning Non-Anginal Pain. |
| Assessment | Non-Anginal Pain is typically not associated with reduced blood flow to the heart muscle. |
| What It May Suggest | This finding suggests that the chest discomfort you experienced is unlikely to be directly related to heart disease. |

...

| Test Value | Resting ECG: Normal |
|---|---|
| What It Measures | This test records the electrical signals of your heart while you are at rest. |
| Assessment | A "Normal" resting ECG means that no significant electrical abnormalities in your heart's rhythm or structure were detected. |
| What It May Suggest | This is a positive finding, suggesting healthy electrical activity of your heart at rest and no immediate signs of strain or damage. |

| Test Value | Max Heart Rate: 160 bpm |
|---|---|
| What It Measures | This is the fastest rate your heart achieved during physical exertion in the test. |
| Assessment | Achieving a maximum heart rate of 160 beats per minute during exercise is a good response for your age and indicates your heart responded well to the stress. |
| What It May Suggest | This suggests a healthy cardiovascular response to physical activity and good exercise capacity. |

| Test Value | Exercise Angina: No |
|---|---|
| What It Measures | This indicates whether you experienced chest pain specifically triggered by physical exertion during the test. |
| Assessment | "No" for exercise angina means you did not report chest pain that was brought on or worsened by physical activity. |
| What It May Suggest | This is a reassuring sign, as exertional chest pain often points to blockages in the heart arteries. |

| Test Value | Oldpeak: 0.8 |
|---|---|
| What It Measures | This value refers to the ST depression induced by exercise relative to rest, indicating changes in the electrical activity of your heart during stress. |
| Assessment | An Oldpeak value of 0.8 mm indicates a mild depression of the ST segment during exercise, which is slightly above ideal but often considered within an acceptable range depending on other factors. |
| What It May Suggest | While mild, this can sometimes suggest subtle variations in blood flow to the heart during exertion, though it needs to be interpreted within the context of all your other results. |

| Test Value | ST Slope: Up |
|---|---|
| What It Measures | This describes the direction of the ST segment on your electrocardiogram during the peak of exercise. |
| Assessment | An 'Up' sloping ST segment is generally considered a normal or favorable response during an exercise test. |
| What It May Suggest | This finding is a positive indicator, suggesting that your heart's electrical activity returns to baseline effectively during or after exercise. |

In [60]:
# Test Case Edge: Single Medically Nonsensical Value
inputs_edge = {
    "Age":              45,
    "Sex":              "Male",
    "ChestPainType":    "ATA",
    "RestingBP":        0,         # zero BP = clinically dead (nonsensical)
    "Cholesterol":      210,
    "FastingBS":        100,
    "RestingECG":       "Normal",
    "MaxHR":            150,
    "ExerciseAngina":   "No",
    "Oldpeak":          0.8,
    "STSlope":          "Up"
}
tc4_advisory, tc4_tables = call_T2(inputs_edge)

Your cardiovascular assessment suggests a lower likelihood of heart disease at this time; however, several important values require attention. Your Resting Blood Pressure of 0 mmHg is an invalid reading and must be re-measured immediately to provide an accurate picture, while your total Cholesterol at 210 mg/dL and Fasting Blood Sugar at 100 mg/dL are at borderline levels, suggesting that lifestyle modifications could help maintain these within optimal ranges. Furthermore, the Oldpeak value of 0.8 and the report of Atypical Angina (ATA) indicate specific areas that warrant further discussion with your doctor to fully understand their implications for your heart health.


The following represents more details regarding each of your cardiovascular test values:


| Test Value | Age: 45 years |
|---|---|
| Assessment | This is your current age, a key contextual factor in assessing cardiovascular risk. |

| Test Value | Sex: Male |
|---|---|
| Assessment | This indicates your biological sex, which can influence typical ranges and risk factors for cardiovascular health. |

| Test Value | Chest Pain Type: ATA |
|---|---|
| What It Measures | This describes the nature of any chest discomfort experienced, which can be an important indicator for various heart conditions. |
| Assessment | Your Chest Pain Type is classified as ATA, which stands for Atypical Angina. |
| What It May Suggest | While "atypical," this type of chest pain still warrants careful attention as it can be associated with underlying cardiovascular conditions. |

...

| Test Value | Resting ECG: Normal |
|---|---|
| What It Measures | This records the electrical signals of your heart while you are at rest, helping to detect abnormalities in heart rhythm or structure. |
| Assessment | Your Resting ECG is normal. |
| What It May Suggest | A normal resting ECG suggests that there are no immediate electrical abnormalities in your heart's rhythm or structure when you are not exerting yourself. |

| Test Value | Max Heart Rate: 150 bpm |
|---|---|
| What It Measures | This is the highest heart rate your heart achieved during the stress test or exercise, reflecting its response to physical exertion. |
| Assessment | Your maximum heart rate during the test was 150 bpm. |
| What It May Suggest | This value helps assess your heart's capacity and response to exertion, and for your age, 150 bpm is a reasonable response during physical activity. |

| Test Value | Exercise Angina: No |
|---|---|
| What It Measures | This indicates whether you experienced chest pain or discomfort during physical activity or stress testing. |
| Assessment | You did not experience angina (chest pain) during exercise. |
| What It May Suggest | The absence of exercise-induced chest pain is a positive sign, as angina provoked by exertion can sometimes indicate underlying heart issues. |

| Test Value | Oldpeak: 0.8 |
|---|---|
| What It Measures | This refers to the amount of ST segment depression on an electrocardiogram during exertion, which can indicate reduced blood flow to the heart muscle. |
| Assessment | Your Oldpeak value is 0.8. |
| What It May Suggest | A value of 0.8 suggests a mild to moderate amount of ST depression, which can be a sign of myocardial ischemia (reduced blood flow to the heart) during stress. |

| Test Value | ST Slope: Up |
|---|---|
| What It Measures | This describes the direction of the ST segment on your electrocardiogram during the recovery phase of an exercise test. |
| Assessment | Your ST Slope is "Up." |
| What It May Suggest | An upsloping ST segment is generally considered a normal or less concerning finding compared to a "flat" or "downsloping" segment, but it should always be interpreted in context with other findings like Oldpeak. |

## Evaluation Summary

### Qualitative Evaluation



#### 1. Edge-Case Handling

| Metric | Evaluation |
|---|---|
| Edge-Case Handling | The template handled the implausible RestingBP value of 0 mmHg correctly and explicitly. The interpretation paragraph reflected this, explicitly naming the invalid reading and instructing the patient to have it re-measured before any assessment could be made. Also, in the per-value table, the model identified the anomaly directly, stating that 0 mmHg "is not a biologically possible blood pressure reading" and that it "indicates a data anomaly or an unmeasurable reading". This behaviour is a direct result of the chain-of-thought structure because each value is examined individually in a dedicated reasoning step before the interpretation is written, implausible values cannot be silently absorbed into a generic response. The model is forced to confront and assess each value on its own terms, which surfaces anomalies that a direct response approach would miss.

---

#### 2. Instruction Following

| Sub-category | Evaluation |
|---|---|
| Personalization | The template achieves a high level of personalization through two layers. The first layer is the per-value tables, where every value receives its own dedicated analysis covering what it measures, whether it is healthy or concerning, and what it may suggest ensuring no value is generalized or skipped. The second layer is the interpretation paragraph, which references specific numerical values from the patient's input to summarize the results and most important indicators. TC1 names the exact Resting Blood Pressure of 140 mmHg, Cholesterol of 240 mg/dL, Fasting Blood Sugar of 160 mg/dL, Oldpeak of 2.3, and Flat ST Slope directly in the interpretation. TC2 as well references Resting Blood Pressure of 110 mmHg, Cholesterol of 180 mg/dL, and Fasting Blood Sugar of 80 mg/dL into the interpretation. TC3 names Resting Blood Pressure of 128 mmHg, Cholesterol of 210 mg/dL, Fasting Blood Sugar of 100 mg/dL, and Oldpeak of 0.8, directly in the interpretation. TC4 names the RestingBP of 0 mmHg as an invalid reading, Cholesterol of 210 mg/dL as borderline, and Fasting Blood Sugar of 100 mg/dL as approaching the prediabetes range, Oldpeak of 0.8 and reported Atyplical Angina. No two responses were identical despite sharing the same input structure, confirming the interpretation is consistently driven by the individual patient's values rather than the prediction label alone. |
| Context | The model maintained the cardiovascular interpretation context consistently across all four test cases. It consistently recognized that it was operating within a hospital application serving a health-literate patient who had just received a cardiovascular test result. The reasoning steps in the tables never introduced topics outside the scope of the patient's cardiovascular profile, and the interpretation paragraph in each test case remained focused on the patient's specific clinical picture without drifting into unrelated health topics or general wellness advice. The model also demonstrated awareness of the patient's profile throughout  maintaining the  tone appropriate for a detail-oriented patient who expects reasoning rather than reassurance.|
| Safety — No Diagnosis | The template follows the safety constraint consistently across all four test cases. TC1 uses "suggest an increased likelihood of cardiovascular concerns" rather than stating a diagnosis, TC2 uses "low probability of heart disease", TC3 uses "low probability of significant heart disease", and finally TC4 uses "suggests a lower likelihood of heart disease at this time" which frames the result as a probabilistic assessment. Hedging phrases such as "can be associated with," "may indicate," and "warrants further discussion" appear consistently across all responses, ensuring findings are always presented as possibilities rather than confirmed facts. |

### Quantitative Evaluation

In [75]:
#I stored this in a file in case I needed it, THIS WILL BE DELETED LATER
import pickle

with open("cot_outputs.pkl", "rb") as f:
    saved = pickle.load(f)

tc1_advisory = saved["tc1_advisory"]
tc2_advisory = saved["tc2_advisory"]
tc3_advisory = saved["tc3_advisory"]
tc4_advisory = saved["tc4_advisory"]
tc1_tables   = saved["tc1_tables"]
tc2_tables   = saved["tc2_tables"]
tc3_tables   = saved["tc3_tables"]
tc4_tables   = saved["tc4_tables"]

print("Loaded successfully")

Loaded successfully


In [76]:
#Prepare output for evaluation
import re

def tables_to_text(markdown_text):
    lines = markdown_text.split('\n')
    text_parts = []

    for line in lines:
        # Skip divider rows (---|---|--- etc.)
        if re.match(r'^\s*\|?[-:\s|]+\|?\s*$', line):
            continue
        # Remove leading/trailing pipes and split by pipe
        line = line.strip().strip('|')
        cells = [cell.strip() for cell in line.split('|')]
        for cell in cells:
            if cell and cell.strip():
                text_parts.append(cell.strip())

    return ' '.join(text_parts)

# Combine tables text + advisory for each TC
full_outputs = {
    "TC1 — Heart Disease Detected": tables_to_text(tc1_tables) + " " + tc1_advisory,
    "TC2 — No HD: All Healthy":     tables_to_text(tc2_tables) + " " + tc2_advisory,
    "TC3 — No HD: Borderline":      tables_to_text(tc3_tables) + " " + tc3_advisory,
    "TC4 — Edge Case":              tables_to_text(tc4_tables) + " " + tc4_advisory
}

#### 1. Flesch-Kincaid Readability Score

The Flesch-Kincaid readability score was computed using the `textstat`
library in Python on the output of each test case.

In [60]:
score = evaluate_readability(full_outputs)["average_score"]

Flesch-Kincaid Readability Scores:
--------------------------------------------------
TC1 — Heart Disease Detected: 42.99
TC2 — No HD: All Healthy: 45.31
TC3 — No HD: Borderline: 41.44
TC4 — Edge Case: 40.9

Template Average: 42.66
Target Range: 60-70


| Test Case | Score |
|---|---|
| TC1 — Heart Disease Detected | 42.99 |
| TC2 — No HD: All Healthy | 45.31 |
| TC3 — No HD: Borderline | 41.44 |
| TC4 — Edge Case | 40.90 |
| **Template Average** | **42.66** |

The template achieved an average Flesch-Kincaid readability score of 42.66. The chain-of-thought template, serves a health-literate
patient who has chosen to receive a detailed, step-by-step
analysis of their cardiovascular test results. For this user profile, a
score in the range of 30–50 which corresponds to college-level reading
material is appropriate and realistic. The template's average score
of 42.66 falls within this adjusted range, indicating that the output is
linguistically calibrated to the patient it is designed to serve. The
additional complexity comes from the clinical explanations in the per-value
tables, which are intentionally more detailed to satisfy a user who expects
reasoning rather than reassurance.
> *Flesch Reading Ease: What It Is and Why It Matters*.
> https://contentwriters.com/blog/flesch-reading-ease-what-it-is-and-why-it-matters/

#### 2. Domain Keyword Density

The domain keyword density was computed by inspection for each test case,
identifying cardiovascular-relevant terms present in the output and
counting their occurrences.

In [74]:
results = evaluate_keyword_density(full_outputs)

Domain Keyword Density:
--------------------------------------------------
TC1 — Heart Disease Detected:
  Total Words       : 638
  Domain Keywords   : 66
  Density           : 10.34%

TC2 — No HD: All Healthy:
  Total Words       : 624
  Domain Keywords   : 57
  Density           : 9.13%

TC3 — No HD: Borderline:
  Total Words       : 777
  Domain Keywords   : 60
  Density           : 7.72%

TC4 — Edge Case:
  Total Words       : 770
  Domain Keywords   : 48
  Density           : 6.23%

Template Average Density: 8.36%


| Test Case | Domain Keywords | Total Words | Density |
|---|---|---|---|
| TC1 — Heart Disease Detected | 66 | 638 | 10.34% |
| TC2 — No HD: All Healthy | 57 | 624 | 9.13% |
| TC3 — No HD: Borderline | 60 | 777 | 7.72% |
| TC4 — Edge Case | 48 | 770 | 6.23% |
| **Template Average** | | | **8.36%** |

The template achieves a moderate domain keyword density of 8.36% on average. This is appropriate for the health-literate patient this template serves: a user who is comfortable with cardiovascular concepts such as blood pressure categories, cholesterol levels, and ECG findings, but is not a clinical professional. A higher density would risk producing output that reads as a clinical report rather than a patient-facing interpretation, while a lower density would fail to engage a health-literate reader who expects their specific clinical values to be named and discussed directly.

TC1 produces the highest density at 10.34%, which reflects the complexity of a heart disease detected profile where multiple concerning values all require individual clinical discussion. TC2 produces a higher density than TC3 at 9.13% versus 7.72% respectively, which may appear counterintuitive given that borderline values would seem to require more clinical discussion. However this is explained by the word counts. TC3 generated significantly more total text (777 words versus 624) because borderline cases require more qualifying and contextualizing language to explain why a value is not yet concerning but still worth monitoring. This additional explanatory text increases the total word count without proportionally increasing the number of cardiovascular domain terms, naturally diluting the density. The variation across test cases is therefore clinically meaningful rather than inconsistent. It reflects the model correctly calibrating the depth of cardiovascular discussion to the specific values present in each patient profile.

#### 3. Medical Accuracy

Each clinical claim in the output was verified against established
cardiovascular reference guidelines.



| # | Test Case | Clinical Claim | Correct |
|---|---|---|---|
| 1 | TC1 | Your resting blood pressure of 140 mmHg falls into the Stage 2 Hypertension category. | Yes |
| 2 | TC1 | Your total cholesterol level of 240 mg/dL is considered high. | Yes |
| 3 | TC1 | Your fasting blood sugar of 160 mg/dL is elevated and indicates diabetes. | Yes |
| 4 | TC1 | Your maximum heart rate during the test was 150 beats per minute. This value is often assessed in context with age and exercise capacity; for a 55-year-old, this may be considered a reasonable response to exertion. | No |
| 5 | TC1 | Your Oldpeak value is 2.3, which represents a significant ST depression during exercise. | Yes |
| 6 | TC2 | Your resting blood pressure of 110 mmHg is within the optimal healthy range. | Yes |
| 7 | TC2 | Your cholesterol level of 180 mg/dL is well within the healthy range. | Yes |
| 8 | TC2 | Your fasting blood sugar of 80 mg/dL is within the optimal healthy range. | Yes |
| 9 | TC2 | A maximum heart rate of 175 bpm during exertion is considered a healthy and appropriate response for your age. | No |
| 10 | TC2 | An Oldpeak value of 0.0 indicates no significant ST depression during exercise. | Yes |
| 11 | TC3 | A reading of 128 mmHg is considered elevated blood pressure, meaning it is higher than ideal but not yet in the hypertension range. | Yes |
| 12 | TC3 | A total cholesterol level of 210 mg/dL is considered borderline high, as optimal levels are typically below 200 mg/dL. | Yes |
| 13 | TC3 | A fasting blood sugar level of 100 mg/dL is at the upper end of the normal range and is considered prediabetic.| Yes |
| 14 | TC3 | Achieving a maximum heart rate of 160 beats per minute during exercise is a good response for your age and indicates your heart responded well to the stress.| No |
| 15 | TC3 | An Oldpeak value of 0.8 mm indicates a mild depression of the ST segment during exercise, which is slightly above ideal but often considered within an acceptable range depending on other factors.| Yes |
| 16 | TC4 | Your Resting Blood Pressure is reported as 0 mmHg, which is not a biologically possible blood pressure reading.| Yes |
| 17 | TC4 | Your total Cholesterol is 210 mg/dL, which is considered borderline high.| Yes |
| 18 | TC4 | Your Fasting Blood Sugar is 100 mg/dL, which is at the upper limit of the healthy range. | No |
| 19 | TC4 | Your maximum heart rate during the test was 150 bpm. This value helps assess your heart's capacity and response to exertion, and for your age, 150 bpm is a reasonable response during physical activity.| Yes |
| 20 | TC4 | Your Oldpeak value is 0.8. A value of 0.8 suggests a mild to moderate amount of ST depression, which can be a sign of myocardial ischemia (reduced blood flow to the heart) during stress.| Yes |

> *Understanding Blood Pressure Readings*.
> https://www.heart.org/en/health-topics/high-blood-pressure/understanding-blood-pressure-readings

---

| Test Case | Correct Claims | Total Claims | Accuracy Rate |
|---|---|---|---|
| TC1 | 4 | 5 | 80% |
| TC2 | 4 | 5 | 80% |
| TC3 | 4 | 5 | 80% |
| TC4 | 4 | 5 | 80% |
| **Template Average** | | | **80%** |

---

#### 4. Passive Voice Rate

The passive voice rate was computed using the `spacy` library in Python.
Each sentence in the output was parsed using the `en_core_web_sm` language
model, and sentences containing the `nsubjpass` dependency tag were
identified as passive. Lower is better.

In [62]:
score = evaluate_passive_voice(full_outputs)

Passive Voice Rate:
--------------------------------------------------
TC1 — Heart Disease Detected:
  Total Sentences  : 38
  Passive Sentences: 6
  Passive Rate     : 15.79%

TC2 — No HD: All Healthy:
  Total Sentences  : 37
  Passive Sentences: 3
  Passive Rate     : 8.11%

TC3 — No HD: Borderline:
  Total Sentences  : 33
  Passive Sentences: 6
  Passive Rate     : 18.18%

TC4 — Edge Case:
  Total Sentences  : 37
  Passive Sentences: 6
  Passive Rate     : 16.22%

Template Average Passive Voice Rate: 14.57%


| Test Case | Passive Sentences | Total Sentences | Passive Rate |
|---|---|---|---|
| TC1 — Heart Disease Detected | 6 | 36 | 15.79% |
| TC2 — No HD: All Healthy | 3 | 37 | 8.11% |
| TC3 — No HD: Borderline | 6 | 33 | 18.18% |
| TC4 — Edge Case | 6 | 37 | 16.22% |
| **Template Average** | | | **14.57%** |

The template achieves an average passive voice rate of 14.57%. TC2 performs strongest at 8.11%, reflecting the positive and reassuring tone of a healthy result where active constructions such as "Your cholesterol level of 180 mg/dL is well within the healthy range" come naturally. TC3 show slightly higher rates at 18.18%, which is attributable to the borderline scenario where the model tends to use constructions such as "A total cholesterol level of 210 mg/dL is considered borderline high" when describing clinical assessments, a pattern that is difficult to eliminate entirely when the model is explaining what a measurement is categorized as. Overall the template demonstrates strong active voice usage across the majority of its output, consistent with the goal of producing text that feels direct and personally addressed to the health-literate patient.

## Assumptions and Limitations




---
### Assumptions
**1. Health-Literate Patient Profile**
The template is written for a patient who approaches their results
with curiosity rather than anxiety, and someone familiar with terms like
blood pressure ranges and cholesterol categories. This
assumption shapes every layer of the output, from the vocabulary
used in the tables to the depth of clinical explanation in the
interpretation.

**2. Human-Readable Feature Values**
The template assumes that all clinical feature values are passed in
their original, human-readable categorical and numerical form prior
to any encoding transformations. The chain-of-thought reasoning in
the per-value tables depends on the model being able to interpret
values like ChestPainType = ATA directly. Encoded representations
such as ChestPainType_ATA = 1 carry no natural language meaning and
would produce assessments that are clinically incoherent.

**3. Hospital Mobile Application Deployment**
The template is designed for deployment within a hospital mobile
application serving one patient at a time following their
cardiovascular test. It is further assumed that in a complete
implementation, the per-value tables will be delivered to the
patient as a downloadable PDF rather than rendered inline on the
mobile screen ensuring the interpretation paragraph remains the primary
visible output while the full reasoning is accessible on demand
without compromising the mobile reading experience.

**4. The Patient Will Read the Tables Before Acting**
The template assumes the patient will engage with both layers of the
output, reading the interpretation first for the overall picture, then
consulting the per-value tables to understand the reasoning behind it.
If the patient reads only the interpretation and ignores the tables, they
receive a response equivalent to what a shorter template would have
produced, meaning the additional reasoning depth goes unused and the
template's core value proposition is not realized.

---

### Limitations

**1. Output Length**
Chain-of-Thought produces the longest output of all four templates
by a significant margin. Even with the interpretation paragraph displayed
first, the full set of per-value tables requires substantial scrolling
on a mobile device. The PDF export assumption mitigates this at the
user experience level, but delivering it requires engineering work
that goes beyond the current basic implementation.

**2. Medical Accuracy Dependent on Model Knowledge**
The template achieved 80% medical accuracy across all test cases. Since
the model reasons entirely from its internal training data with no
external validation mechanism, some assessments may not align
precisely with the most current clinical guidelines, particularly
for age-dependent values such as maximum heart rate, where the
appropriate interpretation shifts meaningfully across different
patient profiles.

**3. Structural Dependency Between Tables and Interpretation**
In the other three templates, the model looks at all the patient's
values together and writes a response based on the overall picture.
In Chain-of-Thought, the model must first go through each value one
by one in the tables, and only then write the interpretation based
on what it found. This means the interpretation is completely tied
to what the tables concluded it cannot step back and consider the
broader clinical picture independently. If the individual table
assessments collectively underrepresent the severity of the patient's
condition, the interpretation will reflect that underrepresentation,
with no mechanism to course-correct based on the full set of values
viewed together.

**4. Reasoning Depth Is Uniform Across All Values**
The template instructs the model to apply the same four-step reasoning
structure to every clinically actionable value, regardless of how
significant or straightforward that value is. A normal resting ECG
receives the same table format as a significantly elevated Oldpeak,
even though the former requires very little clinical explanation and
the latter carries much more weight. This uniformity keeps the output
consistent but means the patient must read through reasoning of equal
depth for both minor and major findings, which may dilute the impact
of the most clinically important values.

# Few-Shot Prompting

### Template ID
3

---

### Approach Definition

Few-shot prompting is a prompt engineering technique in which a language model is provided with a small number of labeled examples before being asked to perform a new but similar task. Instead of relying only on a direct instruction, the model learns the expected pattern of input-output behavior from the examples included in the prompt. These examples act as demonstrations that guide the model toward producing outputs in the required format and according to the intended logic (Brown et al., 2020).

---

### Relation to the Medical Field

Few-shot prompting is particularly well suited to the medical domain because clinical reasoning rarely happens in isolation. When a clinician assesses a new cardiovascular patient, they do not interpret the case from scratch. Instead, they draw on previously encountered patients with similar profiles, comparing the current case against familiar patterns before reaching a conclusion. A cardiologist who has seen many patients with high blood pressure, elevated cholesterol, and chest pain will immediately recognize that combination as a risk pattern worth taking seriously, not because of a single number, but because of how the full picture matches cases seen before(Elstein, 2002).

This is exactly how few-shot prompting works(Brown et al., 2020). By placing a small number of labeled patient records inside the prompt, the model is shown how similar cardiovascular profiles have been interpreted in the past. Each example acts as a reference point, guiding the model to recognize patterns across multiple variables rather than reacting to any single measurement in isolation. In that sense, few-shot prompting functions as a lightweight case-based reasoning mechanism, one that mirrors the comparative, pattern-driven logic that underlies real clinical judgment in cardiovascular care.

> Brown, T. et al. (2020). *Language Models are Few-Shot Learners*. NeurIPS.
> https://arxiv.org/abs/2005.14165

> Elstein, A. S. (2002). *Clinical problem solving and diagnostic decision making*.
> https://pmc.ncbi.nlm.nih.gov/articles/PMC1122649/

---

### Intended Use Case

**1- Deployment Environment**

The system is designed to be deployed as a heart disease interpretation tool within a healthcare mobile application. Patients can access it through their smartphones after obtaining their clinical data, either from a hospital test or by manually entering their values into the system. The health overview is generated instantly and displayed on the same screen.



**2- The User**

The primary user of this template is a returning patient who has used the healthcare application before and has already received at least one prior health overview through the system. This user is familiar with the general style, tone, and structure of the generated response, and they return expecting a consistent and recognisable experience.
Unlike a first-time user, this patient does not need the system to introduce the format from the beginning. Instead, they benefit from receiving their health overview in a stable and predictable structure that allows them to compare their current result with previous overviews, notice changes in their condition, and follow their health status over time.

***Medical background:***

No medical expertise is assumed. However, because the user has prior experience with the system, the response does not need to introduce or justify its structure, and can instead focus directly on the interpretation content. The output should avoid complex terminology and focus on simple explanations supported by examples.

***Emotional state:***

The patient may still feel some concern when reviewing a health-related result, but they are likely to approach the output with greater familiarity than a new user. For that reason, the response should remain supportive and clear while focusing on reassurance through consistency and readability.



**3- Inputs**

3.1 The patient heart disease classification result

3.2 The patient clinical feature values




**4- Output**

A structured but concise response that follows the pattern demonstrated in the examples. The output includes:
- A short interpretation of the patient’s condition
- A brief explanation based on the most relevant features
- A clear, simple tips and suggestions

The response should remain consistent with the format shown in the examples, ensuring readability and ease of understanding.

---

### Design Rationale
**Advantage 1 —  Better adaptation to the task:**
Few-shot prompting helps the model adapt to the specific prediction task by showing it exactly how inputs and outputs should look. This is important in the heart disease dataset because the model needs to deal with structured patient attributes such as age, sex, chest pain type, resting blood pressure, cholesterol, fasting blood sugar, and other clinical indicators. A few examples help the model understand how these features are mapped to the target class.(OpenAI, 2023)

**Advantage 2 — Improved consistency of outputs:**
When examples are included in the prompt, the model is more likely to respond in a consistent format. This is useful in a medical application because predictions should be presented clearly and uniformly. For example, if each example follows the same structure of patient data, prediction, and short explanation, the model is more likely to generate outputs that are easier to compare, evaluate, and interpret.

**Advantage 3 — Useful when labeled examples are meaningful:**
Our dataset already contains labeled cases, which makes few-shot prompting a natural choice. Since the target variable indicates the presence or absence of heart disease, example records can be selected directly from the dataset to guide the model. This allows the prompting technique to align closely with the supervised learning setting, where the model benefits from seeing representative examples before handling unseen cases.



> OpenAI. (2023). *Prompt Engineering Guide*.
> https://platform.openai.com/docs/guides/prompt-engineering

---

### Prompt Engineering Techniques Applied

#### Role Assignment

*You are a cardiovascular assistant supporting a heart disease prediction system.*

The role assigned to the model is **cardiovascular assistant**. This was chosen deliberately over alternatives like "cardiovascular analyst" or "health educator." An analyst makes reasoning visible step by step. An educator builds foundational knowledge. An assistant translates a system's output into a personalised, readable overview for a patient who has just received a classification result — which is precisely what this template requires. The word "supporting" also positions the model as secondary to the prediction system, not as an independent clinical authority.

#### Task Definition

*Your task is to generate a personalised health overview for each patient based on their heart disease classification result and their individual clinical feature values.*

This sentence defines the job with precision. **"Classification result and individual clinical feature values"** makes clear that both inputs matter equally, the label alone is not sufficient to produce the overview, and neither are the values in isolation. The contextual information provided within a prompt is a critical design dimension that directly affects the relevance and accuracy of the model's output (Liao et al., 2023).

> Liao, Q.V., Subramonyam, H., Wang, J. and Wortman Vaughan, J. (2023). *Designerly understanding: Information needs for model transparency to support design ideation for AI-powered user experience.* Proceedings of the 2023 CHI Conference on Human Factors in Computing Systems.
> https://doi.org/10.1145/3560815

#### Few-Shot Prompting?

*Follow the response style demonstrated in the examples.*

This single sentence is the activation instruction for few-shot behaviour. It tells the model that the examples in the user prompt are not merely illustrative, they are the authoritative template for wording, structure, and register. Without this explicit pointer, a model may treat the examples as background context rather than as binding stylistic constraints, producing output that is accurate in content but inconsistent in tone or format.

#### Tone Specification

*Use simple patient-friendly language.*

This rule calibrates the vocabulary level the model must write at. The target reader is a returning patient who has prior experience with the system but may still feel some concern when reviewing a health-related result. Clinical shorthand such as "STSlope: Flat" or "Oldpeak" must be translated into plain terms a general patient can understand, not reproduced verbatim from the feature set. The rule permits necessary terms but sets the default register firmly at accessible rather than clinical.


#### Safety Constraints

The system instruction includes a set of explicit negative rules that define what the model must never do, regardless of the patient's feature values or prediction label.

*Do not provide a diagnosis.*

This is the most critical safety boundary in the prompt. This system is a health literacy tool that bridges the gap between a raw classification result and a meaningful interpretation of that assessment, it is not a licensed clinical instrument. A diagnosis carries legal and medical weight this system is not qualified to provide. It requires a trained physician who can examine the patient directly, review their full medical history, and order further investigations. Framing all findings as possibilities rather than certainties is therefore a fundamental safety requirement that must be maintained across every response (Cascella et al., 2024).

*Do not use technical jargon unless necessary.*

This rule prevents the model from drifting back into clinical language when describing specific findings. A model without this constraint will often reproduce feature names directly — writing "your Oldpeak value of 3.0" rather than explaining what that value means in terms the patient can connect to their lived experience. The qualifier "unless necessary" preserves the model's ability to use a term when no plain equivalent exists, without giving it licence to default to jargon throughout.

*Do not mention probabilities, model confidence, or AI limitations.*

Patients interpret uncertainty language differently from clinicians. A phrase like "the model is 78% confident" invites the patient to reason about the remaining 22% in ways that are not clinically productive and may cause unnecessary distress or, conversely, false reassurance. Mentioning AI limitations such as "this result may not be accurate" undermines the patient's ability to act on the overview and shifts focus away from their clinical values onto the mechanics of the system. This rule keeps the overview focused entirely on the patient's cardiovascular picture.

*Do not invent clinical facts beyond the provided prediction and features.*

The model receives only the feature values listed in the input. It must not introduce conditions, risk factors, or measurements that are not derivable from what is given. In a clinical setting, a fabricated fact — even a plausible-sounding one — could mislead a patient or directly contradict findings their doctor holds. This rule is especially important in a few-shot setting because the examples contain rich clinical detail, and a model without this constraint may generalise from example content into the live patient's output, attributing findings to a patient who does not share them.

#### Conditional Response Logic

*Keep the response to one paragraph. The response should include: (1) a brief interpretation of the result, (2) a short explanation based on the most relevant clinical features, (3) a practical and reassuring overview.*

Two instructions are combined here. The first is a length and display constraint: a single, well-structured paragraph is immediately readable on a mobile screen without scrolling and prevents the model from expanding into unsolicited sub-topics or producing a response that feels overwhelming to a patient who may already be anxious about their result. The second defines the internal architecture that paragraph must follow. All three parts must be present in every response regardless of the prediction label or feature profile. The first part grounds the patient in what the overall result suggests. The second part connects that result to the specific values driving it, ensuring the overview is genuinely personalised rather than label-driven. The third part closes with actionable wellness guidance tied to those same values, so the patient leaves the overview with something concrete to act on. Together, these two instructions ensure the model both stays within the required length and fills that length with the right content in the right order, rather than enumerating every feature indiscriminately.

### Prompt Structure

In [6]:


SYSTEM_T3 = """
You are a cardiovascular assistant supporting a heart disease prediction system.

Your task is to explain cardiovascular risk results to patients in a clear, concise, and supportive manner.
Follow the response style demonstrated in the examples.

Rules:
- Use simple patient-friendly language.
- Do not use technical jargon unless necessary.
- Do not mention probabilities, model confidence, or AI limitations.
- Do not provide a diagnosis.
- Do not invent clinical facts beyond the provided prediction and features.
- Keep the response to one paragraph.
- The response should include:
  1) a brief interpretation of the result,
  2) a short explanation based on the most relevant clinical features,
  3)  a practical and reassuring wellness guidance.
"""

USER_T3 = """
Below are example patient cases and their corresponding responses.
Use them as a guide for wording, structure, tone, and level of detail.

Example 1
Input:
Prediction: No Heart Disease
Age: 42
Sex: Female
ChestPainType: NAP
RestingBP: 118
Cholesterol: 205
FastingBS: 95
RestingECG: Normal
MaxHR: 170
ExerciseAngina: No
Oldpeak: 0.3
STSlope: Up

Output:
The result suggests that there is no current strong indication of heart disease. Your overall clinical profile appears reassuring, particularly with a normal ECG, no exercise-related chest discomfort, and a strong heart rate response during activity. These factors are commonly associated with a lower cardiovascular risk profile. However, it is still important to maintain healthy habits such as regular exercise, balanced nutrition, and routine medical checkups to support long-term heart health.

Example 2
Input:
Prediction: Heart Disease
Age: 63
Sex: Male
ChestPainType: ASY
RestingBP: 155
Cholesterol: 270
FastingBS: 150
RestingECG: ST
MaxHR: 115
ExerciseAngina: Yes
Oldpeak: 3.0
STSlope: Flat

Output:
The result suggests an increased likelihood of heart disease, indicating that your cardiovascular health may require closer medical attention. Several factors may have contributed to this result, including elevated blood pressure, high cholesterol levels, exercise-related chest discomfort, and signs of stress on the heart during activity. These patterns are often associated with higher cardiovascular risk. It is strongly recommended that you consult a healthcare professional for further evaluation and follow appropriate guidance on lifestyle changes and risk management.

Example 3
Input:
Prediction: No Heart Disease
Age: 54
Sex: Male
ChestPainType: ATA
RestingBP: 138
Cholesterol: 245
FastingBS: 130
RestingECG: Normal
MaxHR: 148
ExerciseAngina: No
Oldpeak: 1.8
STSlope: Flat

Output:
The result indicates no current strong sign of heart disease, but some of your clinical values suggest that caution may still be needed. While there is no exercise-related angina and your ECG appears normal, other factors such as moderately elevated cholesterol, increased blood pressure, and higher stress-related indicators may still influence your overall cardiovascular risk. It would be beneficial to monitor your condition regularly and adopt healthy lifestyle practices to reduce potential future risk.

Now write a response for the following patient case in the same style.

Input:
Prediction: {prediction}
Age: {Age}
Sex: {Sex}
ChestPainType: {ChestPainType}
RestingBP: {RestingBP}
Cholesterol: {Cholesterol}
FastingBS: {FastingBS}
RestingECG: {RestingECG}
MaxHR: {MaxHR}
ExerciseAngina: {ExerciseAngina}
Oldpeak: {Oldpeak}
STSlope: {STSlope}

Output:
"""

from IPython.display import display, Markdown

def run_test_case(case_name, inputs):
    print(case_name)
    result = call_gemini(SYSTEM_T3, USER_T3.format(**inputs))
    display(Markdown(result))


### Input / Output Example

#### Test case 1

In [7]:
# Test Case 1: Heart Disease Detected
inputs_1 = {
    "Age": 55,
    "Sex": "Male",
    "ChestPainType": "ATA",
    "RestingBP": 140,
    "Cholesterol": 240,
    "FastingBS": 160,
    "RestingECG": "Normal",
    "MaxHR": 150,
    "ExerciseAngina": "Yes",
    "Oldpeak": 2.3,
    "STSlope": "Flat"
}
inputs_1["prediction"] = get_prediction(inputs_1)
run_test_case("Test Case 1 — Heart Disease", inputs_1)


Test Case 1 — Heart Disease


The result suggests an increased likelihood of heart disease, indicating that your cardiovascular health may require closer medical attention. Several factors may have contributed to this result, including elevated blood pressure, higher cholesterol, elevated fasting blood sugar, and experiencing chest discomfort during exercise, along with signs of stress on the heart during activity. These patterns are often associated with higher cardiovascular risk. It is strongly recommended that you consult a healthcare professional for further evaluation and follow appropriate guidance on lifestyle changes and risk management.

#### Test case 2

In [8]:
# Test Case 2: No Heart Disease Detected — All Healthy Values
inputs_2 = {
    "Age":              35,
    "Sex":              "Female",
    "ChestPainType":    "NAP",
    "RestingBP":        110,
    "Cholesterol":      180,
    "FastingBS":        80,
    "RestingECG":       "Normal",
    "MaxHR":            175,
    "ExerciseAngina":   "No",
    "Oldpeak":          0.0,
    "STSlope":          "Up"
}
inputs_2["prediction"] = get_prediction(inputs_2)
run_test_case("Test Case 2 — No Heart Disease", inputs_2)


Test Case 2 — No Heart Disease


The result suggests that there is no current strong indication of heart disease. Your clinical profile appears very reassuring, especially with normal blood pressure and cholesterol levels, no exercise-related chest discomfort, a healthy heart rate response during activity, and a normal ECG. These factors are commonly associated with a very low cardiovascular risk profile. It is still important to maintain these healthy habits, including regular exercise, balanced nutrition, and routine medical checkups, to support excellent long-term heart health.

#### Test case 3

In [9]:

# Test Case 3: No Heart Disease Detected — Borderline Values
inputs_3 = {
    "Age":              45,
    "Sex":              "Male",
    "ChestPainType":    "NAP",
    "RestingBP":        128,
    "Cholesterol":      210,
    "FastingBS":        100,
    "RestingECG":       "Normal",
    "MaxHR":            160,
    "ExerciseAngina":   "No",
    "Oldpeak":          0.8,
    "STSlope":          "Up"
}
inputs_3["prediction"] = get_prediction(inputs_3)

run_test_case("Test Case 3 — No Heart Disease with Borderline Values", inputs_3)

Test Case 3 — No Heart Disease with Borderline Values


The result suggests that there is no current strong indication of heart disease. Your overall clinical profile appears reassuring, particularly with a normal ECG, no exercise-related chest discomfort, a good heart rate response during activity, and healthy signs during stress on the heart. These factors are commonly associated with a lower cardiovascular risk profile. However, it is still important to maintain healthy habits such as regular exercise, balanced nutrition, and routine medical checkups to support long-term heart health.

#### Test case 4

In [11]:
# Test Case Edge: Single Medically Nonsensical Value
inputs_edge = {
    "Age":              45,
    "Sex":              "Male",
    "ChestPainType":    "ATA",
    "RestingBP":        0,         # zero BP = clinically dead (nonsensical)
    "Cholesterol":      210,
    "FastingBS":        100,
    "RestingECG":       "Normal",
    "MaxHR":            150,
    "ExerciseAngina":   "No",
    "Oldpeak":          0.8,
    "STSlope":          "Up"
}
inputs_edge["prediction"] = get_prediction(inputs_edge)
run_test_case("Test Case 4 — Edge Case", inputs_edge)

Test Case 4 — Edge Case


The result suggests that there is no current strong indication of heart disease. Your overall clinical profile appears reassuring, particularly with a normal ECG, no exercise-related chest discomfort, and a good heart rate response and favorable heart activity pattern during exercise. These factors are commonly associated with a lower cardiovascular risk profile. However, it is still important to maintain healthy habits such as regular exercise, balanced nutrition, and routine medical checkups to support long-term heart health.

## Evaluation Summary

### Comparative Analysis


##### Qualitative Metrics

**1. Edge-Case Handling**
| Metric | Evaluation |
|---|---|
| Edge-Case Handling | The template does not handle the medically implausible value in TC4 effectively. Although the input contains a RestingBP value of 0 mmHg, which is clinically nonsensical, the response does not flag it as abnormal or impossible. Instead, it proceeds with a fully reassuring interpretation and presents the overall profile as low-risk. This indicates that the template is not robust in detecting invalid or unrealistic clinical inputs. Such behavior may lead to misleading conclusions in safety-sensitive contexts, as clearly incorrect data is treated as normal rather than being questioned or highlighted.|

---

**2. Instruction Following**


| Sub-category | Description |
|---|---|
| Personalization | The outputs clearly adapt to individual patient profiles rather than relying solely on the prediction label. In Test Case 1 (Heart Disease), the response highlights contributing factors such as elevated blood sugar, high blood pressure, cholesterol, and exercise-related chest discomfort. In Test Case 3, despite the “No Heart Disease” label, the model still identifies borderline values (cholesterol and fasting blood sugar) and reflects them in the explanation. Similarly, in Test Case 2, the response emphasizes healthy indicators such as normal ECG and absence of exercise-induced symptoms. This demonstrates that the model integrates multiple clinical features when generating responses, rather than being purely label-driven.|
| Context | The model maintains the intended cardiovascular interpretation context consistently across all four test cases. In every output, the response remains focused on heart-health-related findings, cardiovascular risk, and patient-facing medical guidance. There is no observable drift into unrelated advice, general chatbot behaviour, or off-topic explanation. |
| Safety — No Diagnosis | The template successfully adheres to the safety constraint of avoiding diagnosis. In all outputs, the model uses cautious phrasing such as “suggests” and frames the result as a possibility rather than a confirmed medical condition. It also consistently recommends consulting a healthcare professional in higher-risk cases (Test Case 1), which reinforces safe usage without overstepping clinical boundaries. |





##### Quantitative Metrics

#### 1. Flesch-Kincaid Readability Score



In [ ]:
import nltk
import textstat

nltk.download('cmudict')

# T1 outputs
outputs_1 = {
    "TC1 — Heart Disease Detected": """The result suggests an increased likelihood of heart disease, indicating that your cardiovascular health may require closer medical attention. Several factors may have contributed to this result, including elevated blood pressure, higher cholesterol, elevated fasting blood sugar, and experiencing chest discomfort during exercise, along with signs of stress on the heart during activity. These patterns are often associated with higher cardiovascular risk. It is strongly recommended that you consult a healthcare professional for further evaluation and follow appropriate guidance on lifestyle changes and risk management.""",

    "TC2 — No Heart Disease": """The result suggests that there is no current strong indication of heart disease. Your clinical profile appears very reassuring, especially with normal blood pressure and cholesterol levels, no exercise-related chest discomfort, a healthy heart rate response during activity, and a normal ECG. These factors are commonly associated with a very low cardiovascular risk profile. It is still important to maintain these healthy habits, including regular exercise, balanced nutrition, and routine medical checkups, to support excellent long-term heart health.""",

    "TC3 — No Heart Disease with Borderline Values": """The result suggests that there is no current strong indication of heart disease. Your overall clinical profile appears reassuring, particularly with a normal ECG, no exercise-related chest discomfort, a good heart rate response during activity, and healthy signs during stress on the heart. These factors are commonly associated with a lower cardiovascular risk profile. However, it is still important to maintain healthy habits such as regular exercise, balanced nutrition, and routine medical checkups to support long-term heart health.""",

    "TC4 — Edge Case": """The result suggests that there is no current strong indication of heart disease. Your overall clinical profile appears reassuring, particularly with a normal ECG, no exercise-related chest discomfort, and a good heart rate response and favorable heart activity pattern during exercise. These factors are commonly associated with a lower cardiovascular risk profile. However, it is still important to maintain healthy habits such as regular exercise, balanced nutrition, and routine medical checkups to support long-term heart health."""
}

results_readability_1 = evaluate_readability(outputs_1)

Flesch-Kincaid Readability Scores:
--------------------------------------------------
TC1 — Heart Disease Detected: 10.09
TC2 — No Heart Disease: 23.27
TC3 — No Heart Disease with Borderline Values: 27.6
TC4 — Edge Case: 21.69

Template Average: 20.66


[nltk_data] Downloading package cmudict to /Users/janaa/nltk_data...
[nltk_data]   Package cmudict is already up-to-date!


| Test Case | Score |
|---|---|
| TC1 — Heart Disease Detected | 10.09 |
| TC2 — No HD: All Healthy | 23.27 |
| TC3 — No HD: Borderline | 27.6 |
| TC4 — Edge Case | 21.69 |
| **Template Average** | **20.66** |


The template achieved an average Flesch-Kincaid readability score of 20.66, indicating that the responses are relatively difficult to read and may require more effort to understand. This level of complexity is generally suitable for the intended user, who is a returning patient familiar with the system and its response style.
The lowest score appears in TC1, suggesting that responses describing higher-risk conditions tend to be more complex, likely due to longer sentences and more detailed explanations.
Overall, while the outputs are clear and informative, the readability level may still make them harder to follow for less experienced users, but remains appropriate for users who are already familiar with the system.

#### 2. Domain Keyword Density




In [18]:
# T1 outputs
outputs_1 = {
    "TC1 — Heart Disease Detected": """The result suggests an increased likelihood of heart disease, indicating that your cardiovascular health may require closer medical attention. Several factors may have contributed to this result, including elevated blood pressure, higher cholesterol, elevated fasting blood sugar, and experiencing chest discomfort during exercise, along with signs of stress on the heart during activity. These patterns are often associated with higher cardiovascular risk. It is strongly recommended that you consult a healthcare professional for further evaluation and follow appropriate guidance on lifestyle changes and risk management.""",

    "TC2 — No Heart Disease": """The result suggests that there is no current strong indication of heart disease. Your clinical profile appears very reassuring, especially with normal blood pressure and cholesterol levels, no exercise-related chest discomfort, a healthy heart rate response during activity, and a normal ECG. These factors are commonly associated with a very low cardiovascular risk profile. It is still important to maintain these healthy habits, including regular exercise, balanced nutrition, and routine medical checkups, to support excellent long-term heart health.""",

    "TC3 — No Heart Disease with Borderline Values": """The result suggests that there is no current strong indication of heart disease. Your overall clinical profile appears reassuring, particularly with a normal ECG, no exercise-related chest discomfort, a good heart rate response during activity, and healthy signs during stress on the heart. These factors are commonly associated with a lower cardiovascular risk profile. However, it is still important to maintain healthy habits such as regular exercise, balanced nutrition, and routine medical checkups to support long-term heart health.""",

    "TC4 — Edge Case": """The result suggests that there is no current strong indication of heart disease. Your overall clinical profile appears reassuring, particularly with a normal ECG, no exercise-related chest discomfort, and a good heart rate response and favorable heart activity pattern during exercise. These factors are commonly associated with a lower cardiovascular risk profile. However, it is still important to maintain healthy habits such as regular exercise, balanced nutrition, and routine medical checkups to support long-term heart health."""
}

# Evaluate domain keyword density using reference list
results_1 = evaluate_keyword_density(outputs_1)

Domain Keyword Density:
--------------------------------------------------
TC1 — Heart Disease Detected:
  Total Words       : 85
  Domain Keywords   : 6
  Density           : 7.06%

TC2 — No Heart Disease:
  Total Words       : 78
  Domain Keywords   : 5
  Density           : 6.41%

TC3 — No Heart Disease with Borderline Values:
  Total Words       : 78
  Domain Keywords   : 4
  Density           : 5.13%

TC4 — Edge Case:
  Total Words       : 76
  Domain Keywords   : 4
  Density           : 5.26%

Template Average Density: 5.96%


| Test Case | Domain Keywords | Total Words | Density |
|---|---|---|---|
| TC1 — Heart Disease Detected | 6 | 85 | 7.06% |
| TC2 — No HD: All Healthy | 5 | 78 | 6.41% |
| TC3 — No HD: Borderline | 4 | 78 |5.13% |
| TC4 — Edge Case | 4 | 76| 5.26% |
| **Template Average** | | | **5.96%** |

The template achieves a relatively low domain keyword density across all test cases, with an average of 5.96%. This is consistent with the intended user profile of a non-expert but returning patient, as the responses avoid excessive use of clinical terminology.
While key cardiovascular concepts such as blood pressure, cholesterol, and heart-related indicators are still present, they are used sparingly and integrated within clear explanatory language.
This balance supports readability while maintaining consistency in the response structure, making the outputs suitable for users who are already familiar with the system and expect clear but not overly technical explanations.

#### 3. Medical Accuracy


| # | Test Case | Clinical Claim | Correct |
|---|---|---|---|
| 1 | TC1 | Blood pressure is described as high| Yes |
| 2 | TC1 | Cholesterol is described as high/increased | Yes |
| 3 | TC1 | Fasting blood sugar is described as elevated | Yes |
| 4 | TC2 | Blood pressure is described as healthy | Yes |
| 5 | TC2 | Cholesterol is described as healthy | Yes |
| 6 | TC2 | Blood sugar is described as healthy| Yes |
| 7 | TC3 | Heart rate response is described as good| No |
| 8 | TC4 | Heart rate response is described as good| Yes |
---

| Test Case | Correct Claims | Total Claims | Accuracy Rate |
|---|---|---|---|
| TC1 | 3 | 3 | 100% |
| TC2 | 3 | 3 | 100% |
| TC3 | 0 | 1 | 0% |
| TC4 | 1 | 1 | 100% |
| **Template Average** | | | **87.5%** |





---

#### 4. Passive Voice Rate



In [ ]:
# T1 outputs
outputs_1 = {
    "TC1 — Heart Disease Detected": """The result suggests an increased likelihood of heart disease, indicating that your cardiovascular health may require closer medical attention. Several factors may have contributed to this result, including elevated blood pressure, higher cholesterol, elevated fasting blood sugar, and experiencing chest discomfort during exercise, along with signs of stress on the heart during activity. These patterns are often associated with higher cardiovascular risk. It is strongly recommended that you consult a healthcare professional for further evaluation and follow appropriate guidance on lifestyle changes and risk management.""",

    "TC2 — No Heart Disease": """The result suggests that there is no current strong indication of heart disease. Your clinical profile appears very reassuring, especially with normal blood pressure and cholesterol levels, no exercise-related chest discomfort, a healthy heart rate response during activity, and a normal ECG. These factors are commonly associated with a very low cardiovascular risk profile. It is still important to maintain these healthy habits, including regular exercise, balanced nutrition, and routine medical checkups, to support excellent long-term heart health.""",

    "TC3 — No Heart Disease with Borderline Values": """The result suggests that there is no current strong indication of heart disease. Your overall clinical profile appears reassuring, particularly with a normal ECG, no exercise-related chest discomfort, a good heart rate response during activity, and healthy signs during stress on the heart. These factors are commonly associated with a lower cardiovascular risk profile. However, it is still important to maintain healthy habits such as regular exercise, balanced nutrition, and routine medical checkups to support long-term heart health.""",

    "TC4 — Edge Case": """The result suggests that there is no current strong indication of heart disease. Your overall clinical profile appears reassuring, particularly with a normal ECG, no exercise-related chest discomfort, and a good heart rate response and favorable heart activity pattern during exercise. These factors are commonly associated with a lower cardiovascular risk profile. However, it is still important to maintain healthy habits such as regular exercise, balanced nutrition, and routine medical checkups to support long-term heart health."""
}

results_passive_1 = evaluate_passive_voice(outputs_1)

Passive Voice Rate:
--------------------------------------------------
TC1 — Heart Disease Detected:
  Total Sentences  : 4
  Passive Sentences: 2
  Passive Rate     : 50.0%

TC2 — No Heart Disease:
  Total Sentences  : 4
  Passive Sentences: 1
  Passive Rate     : 25.0%

TC3 — No Heart Disease with Borderline Values:
  Total Sentences  : 4
  Passive Sentences: 1
  Passive Rate     : 25.0%

TC4 — Edge Case:
  Total Sentences  : 4
  Passive Sentences: 1
  Passive Rate     : 25.0%

Template Average Passive Voice Rate: 31.25%


| Test Case | Passive Sentences | Total Sentences | Passive Rate |
|---|---|---|---|
| TC1 — Heart Disease Detected | 2 | 4 | 50.0% |
| TC2 — No HD: All Healthy | 1 | 4 | 25.0% |
| TC3 — No HD: Borderline | 1 | 4 | 25.0% |
| TC4 — Edge Case | 1 | 4 | 25.0% |
| **Template Average** | | | **31.25%** |

The template shows moderate but somewhat inconsistent passive voice usage across test cases. TC1 has the highest passive rate at 50%, which may make the response feel more formal and less directly engaging for the patient.
TC2, TC3, and TC4 each have a passive rate of 25%, reflecting a mix of active and indirect phrasing across most responses.
The overall template average of 31.25% suggests that passive voice is used fairly often, which may slightly reduce clarity and directness. While the responses remain understandable, more consistent use of active voice could improve clarity and make the tone more engaging and patient-friendly.

## Assumptions and Limitations

### Assumptions

**1.  Returning Patient Profile**
The patient is assumed to have prior experience with the system and to be familiar with its response structure and tone. The prompt, format, and language were all calibrated around this assumption. If deployed for first-time users or patients with no prior frame of reference, the template would require significant adaptation to reintroduce the format and provide additional contextual explanation.


**2. Representativeness of Few-Shot Examples**
The three examples embedded in the prompt are assumed to be sufficiently representative of the range of patient profiles the model may encounter. In practice, the examples cover clearly positive and clearly negative cases only, which limits the model's guidance when handling borderline or mixed-indicator profiles.


---

### Limitations

**1. Edge-Case Handling**
The template does not handle clinically implausible input values. As
demonstrated in TC4, a RestingBP of 0 mmHg was not flagged as impossible
and the model proceeded to generate a confident response. مكررة من ليان

**2. Example Sensitivity**
Few-shot performance is inherently sensitive to the selection, order,
and phrasing of the provided examples. The three examples in this template
were chosen to represent distinct clinical profiles, but small changes
to their wording or structure could meaningfully shift the model's output
style and content consistency across test cases.

**3. Risk of Over-Generalisation from Examples**
Despite the explicit constraint against inventing clinical facts,
the richness of the few-shot examples may still lead the model to
attribute findings from example cases to a live patient who does not
share them. This risk is particularly relevant when a new patient's profile partially resembles an example but differs on key indicators.



# Zero-Shot Prompting

### Template ID
4

---

### Approach Definition

Zero-shot prompting is a prompt engineering technique in which a language model is asked to perform a task using only a direct instruction, without being given any examples of the expected input-output pattern beforehand. Instead of learning from demonstrations included in the prompt, the model relies entirely on its pretrained language understanding and the clarity of the instruction to generate an appropriate response (Kojima et al. 2022).

> Kojima, T. et al. (2022). *Large Language Models are Zero-Shot Reasoners*.
> https://arxiv.org/abs/2205.11916
---

### Relation to the Medical Field

Zero-shot prompting is relevant to the medical field because many healthcare applications need to generate useful responses from clinical information in a straightforward way. In medical settings, the ability to give clear output from a direct instruction is valuable, especially when the task requires the model to respond in a controlled format. This makes zero-shot prompting suitable for health-related systems that depend on consistency, clarity, and simple communication with users. In our project, zero-shot prompting fits the heart disease interpretation context because the system needs to produce a short and understandable message based on the patient’s result and clinical values without relying on example cases inside the prompt. This allows the prompt to remain simple while still guiding the model to generate an appropriate response (Sivarajkumar et al., 2023).

> Sivarajkumar, S. et al. (2023). *HealthPrompt: A Zero-shot Learning Paradigm for Clinical Natural Language Processing*.
> https://pmc.ncbi.nlm.nih.gov/articles/PMC10148337/

---

### Intended Use Case

**1- Deployment Environment**

The system is designed to be deployed as a heart disease interpretation tool within a healthcare mobile application. Patients can access it through their smartphones after obtaining their clinical data, either from a hospital test or by manually entering their values into the system. The advice is generated instantly and displayed on the same screen.


**2- The User**

The primary user of this template is a patient who is likely to feel anxious when receiving health-related information and prefers communication that gives immediate reassurance. They are not looking for a deep or highly detailed explanation. Instead, they want to quickly understand the situation in simple language and feel that they are being guided clearly and gently. This user is likely to focus first on the overall meaning of the result rather than the technical details behind it. They may also read the message quickly, especially if they are worried, so information that is too dense or too long can make understanding harder.

***Medical background:***

No medical expertise is assumed. Health-related terms, measurements, and clinical concepts are understood only at a general everyday level, not at a professional or technical level.

***Emotional state:***

The patient may be worried and needs reassurance early. They are more likely to respond well to language that feels calm, supportive, and steady, and less well to wording that sounds harsh, alarming, or overly certain.


**3- Inputs**

3.1 The patient heart disease classification result

3.2 The patient clinical feature values


**4- Output**

A short personalised health overview written in clear, plain, and supportive language. It briefly presents the patient’s result, highlights only the most relevant concerning or borderline values in an accessible way, and keeps the message calm, focused, and easy to understand. The overview does not list all feature values, does not explain the reasoning process, and remains brief enough to read comfortably on a mobile screen without scrolling.

---

### Design Rationale
**Advantage 1 — Simple prompt design:**
Zero-shot prompting does not require additional example data inside the prompt, which makes it useful when supporting examples are limited, unnecessary, or difficult to prepare. In our case, the model will only need a clear instruction to generate the personalised health overview, so the template remains lightweight and practical. This reduces preparation effort and makes the approach easier to apply consistently across many patient cases (IBM, 2025).

**Advantage 2 — Speed and efficiency:**
Since zero-shot prompting does not require the model to process example cases before generating a response, it can produce outputs more quickly and with less prompt overhead. In our project, this is useful because the system is designed to generate a short personalised health overview directly from the patient’s result and clinical values (Shah, 2024).

**Advantage 3 — Flexiblity:**
Zero-shot prompts are easy to adapt when the wording, tone, or output requirements need to change. Since the template is not tied to fixed demonstrations, updating it requires little effort compared with prompts that depend on carefully chosen examples. This makes zero-shot prompting a flexible option for our system (IBM, 2025).

> IBM. (2025). *What is zero-shot prompting?*.
> https://www.ibm.com/think/topics/zero-shot-prompting

> Shah, D. (2024). Zero-Shot vs. Few-Shot Prompting: Choosing the Right Approach for Your AI Model.
> https://portkey.ai/blog/zero-shot-vs-few-shot-prompting/
---
### Prompt Engineering Techniques Applied

#### Role Assignment
*You are a cardiovascular health assistant deployed in a hospital mobile application.*

The system instruction begins by assigning the model a specific role as a cardiovascular health analyst deployed in a hospital mobile application. This role anchors the model’s language, tone, and domain focus within a realistic patient-care setting, helping it respond as a health-focused assistant rather than as a general-purpose chatbot. This is especially important in our context because the target user is a patient who may feel anxious when receiving health-related information and is not looking for a detailed technical explanation. Instead, the user wants immediate reassurance, simple meaning, and clear guidance. By defining the role clearly at the start, the prompt helps the model generate an output that matches this need.

#### Task Definition
*Your task is to generate a short personalised health overview for a patient based on their heart disease classification result and individual clinical feature values.*

The system prompt clearly defines the exact task the model must perform, which is generating a short personalised health overview based on the patient’s classification result and clinical feature values. This helps reduce ambiguity and ensures that the model focuses on the intended objective instead of drifting into broad health explanation or general advice. In a zero-shot setting, this is especially important because the model relies entirely on direct instruction rather than examples (Liao et al., 2023).

> Liao, Q.V., Subramonyam, H., Wang, J. and Wortman Vaughan, J. (2023). *Designerly understanding: Information needs for model transparency to support design ideation for AI-powered user experience.* Proceedings of the 2023 CHI Conference on Human Factors in Computing Systems.
> https://doi.org/10.1145/3560815

#### User Context Priming
*The user is a patient with no medical background who may feel worried and needs immediate reassurance, simple meaning, and brief guidance.*

The system prompt explicitly describes the target user as a patient with no medical background who may feel worried and needs immediate reassurance, simple meaning, and brief guidance. This primes the model to adapt both its tone and level of explanation to the patient’s likely needs. As a result, the generated response is more likely to feel accessible, supportive, and appropriate for a non-expert audience.

#### Tone Specification
As previously mentioned, the user is likely to feel anxious, and prefers communication that gives immediate reassurance. The following instructions were written to match the needs of the target user.

*Use clear, plain, patient-friendly language.*

This guides the model to choose simple and familiar wording that the user can understand quickly and without extra effort. Since the user wants simple meaning rather than complex explanation, plain language helps the overview feel more direct, accessible, and immediately useful. It also makes the message easier to process at a glance, which is important when the user may already feel anxious or distracted.

*Do not use technical jargon, abbreviations, or hard-to-understand clinical terms.*

This keeps the response from becoming difficult to follow or mentally demanding. Because the user is not looking for technical detail, avoiding specialised terms helps the message stay easy to understand and prevents confusion from getting in the way of reassurance. It also supports a smoother reading experience by removing words that might interrupt understanding or make the message feel too formal.

*Keep the tone warm, reassuring, supportive, gentle, and easy to understand.*

This shapes the response so that it feels emotionally safe and steady for the user. Since the user needs immediate reassurance and clear guidance, a warm and supportive tone helps the message feel more comforting, approachable, and easier to receive. It also makes the overview feel less mechanical, which can help the user trust the message and stay calm while reading it.

*Present health-related concerns in a calm and balanced way that informs the user without causing unnecessary fear.*

This helps the model communicate important information without making the message feel heavy or distressing. Because the user wants clear guidance rather than emotional intensity, this instruction encourages a balanced style that informs while still preserving reassurance. It also helps the response acknowledge concern in a measured way, so the user can understand the issue without feeling pushed into panic.


*Avoid alarming, harsh, dramatic, or overly serious wording.*

This reduces the chance that the user will feel more stressed after reading the overview. Since the user may already feel uneasy, avoiding strong or frightening wording helps keep attention on understanding the message rather than reacting emotionally to it. It also helps the overview maintain a softer and more controlled tone, which better supports the user’s need for reassurance.

*Do not begin the response with greetings such as “Hello”, “Hi”, or any salutation.*

This keeps the response direct and focused on the information the user actually needs. Because the user wants quick meaning and immediate guidance, removing greetings helps the overview start right away with useful content instead of unnecessary filler. It also saves space and supports the goal of keeping the message short, efficient, and easy to read.


#### Safety Constraints
*Do not mention probabilities, model confidence, AI, or model limitations.*

This prevents the response from introducing technical details about the underlying system that are not useful to the patient and may reduce clarity or trust. Information about confidence scores, AI behaviour, or model limitations can distract from the patient’s actual health overview and make the message feel more mechanical than supportive. By excluding these details, the prompt keeps the response centred on the patient’s condition in a way that feels more human, relevant, and easier to understand.

*Do not provide a diagnosis.*

This keeps the overview within a safe interpretation role and avoids presenting the generated response as a formal medical conclusion. In a healthcare context, a diagnosis carries strong clinical authority and should not be stated casually by a generated message. This rule helps ensure that the model remains within its intended function, which is to interpret the provided result in accessible language rather than act as a substitute for professional medical judgment.

*Do not invent facts beyond the provided prediction and features.*

This ensures that the model stays grounded in the actual patient data and does not add unsupported or imagined information that could mislead the user. When a response includes details that were not present in the input, it risks sounding confident while being inaccurate, which is especially dangerous in a medical context. This rule protects factual reliability by forcing the model to remain anchored to the available case information and nothing more.

#### Conciseness Constraint
*Do not list all features.*

This prevents the response from turning into a full summary of every input value and helps keep the overview focused on only the most meaningful information. Listing every feature would make the output longer, denser, and harder to process, especially for a patient who wants quick clarity rather than a complete data review. By narrowing the content, this instruction encourages the model to select the details that matter most and present them in a more useful and digestible way.

*Do not explain the reasoning process.*

This helps keep the overview focused on what the user actually needs, which is a clear and simple understanding of the result rather than a detailed breakdown of how it was interpreted. Since the user is looking for immediate reassurance, simple meaning, and clear guidance, step-by-step reasoning can make the message feel longer, heavier, and more difficult to follow.

*Focus only on the most relevant concerning or borderline values.*

This helps the model identify and select the most important details from the patient’s case so the overview remains targeted, useful, and easy to understand. Instead of spreading attention across every variable, the prompt directs the model toward the values that are most likely to matter for the patient’s current health picture. This improves clarity by making the message more focused and ensures that the patient leaves with a better understanding of the key points rather than a scattered impression of many small details.

#### Output Length Constraint
*Keep the response brief and clear.*

This helps the model produce a message that matches the needs of the intended user. The user may read quickly and may already feel uneasy, a brief and clear response is easier to process and less likely to feel overwhelming. This rule encourages the model to priorities clarity, relevance, and simplicity so the user can understand the message without extra effort.

*Keep the full response short enough to read comfortably on a mobile screen without scrolling.*

This aligns the generated overview with the intended user experience by encouraging concise communication that feels practical and easy to read in a mobile setting. Since many patients may view the result on a phone, this rule helps the model produce content that fits naturally within that environment and does not feel visually heavy. It also reinforces the broader design goal of making the overview feel quick, accessible, and comfortable to read in one glance.

### Prompt Structure

In [86]:
SYSTEM_T4 = """
You are a cardiovascular health analyst deployed in a hospital mobile application.

Your task is to generate a short personalised health overview for a patient based on their heart disease classification result and individual clinical feature values.

The user is a patient with no medical background who may feel worried and needs immediate reassurance, simple meaning, and brief guidance.

Rules:
- Use clear, plain, patient-friendly language, but do not begin with greetings or salutations.
- Keep the tone warm, reassuring, supportive, gentle, and easy to understand.
- Present health-related concerns in a calm and balanced way that informs the user without causing unnecessary fear.
- Avoid alarming, harsh, dramatic, or overly serious wording.
- Do not use technical jargon, abbreviations, or hard-to-understand clinical terms.
- Do not begin the response with greetings such as "Hello", "Hi", or any salutation.
- Do not mention probabilities, model confidence, AI, or model limitations.
- Do not provide a diagnosis.
- Do not invent facts beyond the provided prediction and features.
- Do not explain the reasoning process.
- Do not list all features.
- Focus only on the most relevant concerning or borderline values.
- Keep the response brief and clear.
- Keep the full response short enough to read comfortably on a mobile screen without scrolling.
"""

USER_T4 = """
Write a personalised cardiovascular health overview for the following patient case.

Prediction: {prediction}
Age: {Age}
Sex: {Sex}
ChestPainType: {ChestPainType}
RestingBP: {RestingBP}
Cholesterol: {Cholesterol}
FastingBS: {FastingBS}
RestingECG: {RestingECG}
MaxHR: {MaxHR}
ExerciseAngina: {ExerciseAngina}
Oldpeak: {Oldpeak}
STSlope: {STSlope}
"""

from IPython.display import display, Markdown

def run_zero_shot(inputs):
    full_inputs = inputs.copy()
    full_inputs["prediction"] = get_prediction(full_inputs)

    result = call_gemini(SYSTEM_T4, USER_T4.format(**full_inputs))
    display(Markdown(result))
    return result

#### Test case 1

In [91]:
# Test Case 1: Heart Disease Detected
inputs_1 = {
    "Age": 55,
    "Sex": "Male",
    "ChestPainType": "ATA",
    "RestingBP": 140,
    "Cholesterol": 240,
    "FastingBS": 160,
    "RestingECG": "Normal",
    "MaxHR": 150,
    "ExerciseAngina": "Yes",
    "Oldpeak": 2.3,
    "STSlope": "Flat"
}

run_zero_shot(inputs_1)


Based on your information, it suggests your heart health needs a closer look. We noticed your blood pressure is a bit higher than ideal, and your cholesterol and blood sugar levels are elevated. You also mentioned experiencing some chest discomfort during exercise, and tests showed your heart working harder during physical activity.

These findings are important to address. We strongly recommend discussing this overview with your doctor soon. They can help you understand these points better and create a personalized plan to support your heart health, possibly including lifestyle adjustments and other steps.

'\nBased on your information, it suggests your heart health needs a closer look. We noticed your blood pressure is a bit higher than ideal, and your cholesterol and blood sugar levels are elevated. You also mentioned experiencing some chest discomfort during exercise, and tests showed your heart working harder during physical activity.\n\nThese findings are important to address. We strongly recommend discussing this overview with your doctor soon. They can help you understand these points better and create a personalized plan to support your heart health, possibly including lifestyle adjustments and other steps.'

#### Test case 2

In [92]:
# Test Case 2: No Heart Disease Detected — All Healthy Values
inputs_2 = {
    "Age":              35,
    "Sex":              "Female",
    "ChestPainType":    "NAP",
    "RestingBP":        110,
    "Cholesterol":      180,
    "FastingBS":        80,
    "RestingECG":       "Normal",
    "MaxHR":            175,
    "ExerciseAngina":   "No",
    "Oldpeak":          0.0,
    "STSlope":          "Up"
}

run_zero_shot(inputs_2)

Your current health profile is very encouraging, suggesting that you do not show signs of heart disease. Your blood pressure, cholesterol, and blood sugar levels are all looking good. This is a wonderful foundation for your heart health. Continue with your healthy habits to keep your heart strong.

'Your current health profile is very encouraging, suggesting that you do not show signs of heart disease. Your blood pressure, cholesterol, and blood sugar levels are all looking good. This is a wonderful foundation for your heart health. Continue with your healthy habits to keep your heart strong.'

#### Test case 3

In [110]:
# Test Case 3: No Heart Disease Detected — Borderline Values
inputs_3 = {
    "Age":              45,
    "Sex":              "Male",
    "ChestPainType":    "NAP",
    "RestingBP":        128,
    "Cholesterol":      210,
    "FastingBS":        100,
    "RestingECG":       "Normal",
    "MaxHR":            160,
    "ExerciseAngina":   "No",
    "Oldpeak":          0.8,
    "STSlope":          "Up"
}

run_zero_shot(inputs_3)

Your recent heart health assessment is reassuring, indicating you do not show signs of heart disease at this time.

To support your long-term well-being, we noticed a few areas that could benefit from your attention. Your blood pressure and cholesterol levels are a little higher than ideal. Additionally, your fasting blood sugar is at a level where it's good to keep an eye on it.

Embracing a healthy lifestyle with a balanced diet and regular activity is a wonderful way to care for your heart. We recommend discussing these points with your doctor for personalized guidance.

"Your recent heart health assessment is reassuring, indicating you do not show signs of heart disease at this time.\n\nTo support your long-term well-being, we noticed a few areas that could benefit from your attention. Your blood pressure and cholesterol levels are a little higher than ideal. Additionally, your fasting blood sugar is at a level where it's good to keep an eye on it.\n\nEmbracing a healthy lifestyle with a balanced diet and regular activity is a wonderful way to care for your heart. We recommend discussing these points with your doctor for personalized guidance."

In [18]:
# Test Case Edge: Single Medically Nonsensical Value
inputs_edge = {
    "Age":              45,
    "Sex":              "Male",
    "ChestPainType":    "ATA",
    "RestingBP":        0,         # zero BP = clinically dead (nonsensical)
    "Cholesterol":      210,
    "FastingBS":        100,
    "RestingECG":       "Normal",
    "MaxHR":            150,
    "ExerciseAngina":   "No",
    "Oldpeak":          0.8,
    "STSlope":          "Up"
}

run_zero_shot(inputs_edge)

Based on the information, your heart health outlook appears reassuring, indicating no heart disease at this time. It's great that your heart's electrical activity is normal and you don't experience chest discomfort with exercise.

We noticed your cholesterol level of 210 is a little higher than ideal, and your fasting blood sugar of 100 is at the upper end of the normal range. It's also a good idea to ensure your blood pressure is regularly checked and within a healthy range.

Focusing on a balanced diet, regular physical activity, and routine check-ups with your doctor can help keep your heart healthy and manage these numbers effectively. Your doctor can offer personalized advice and support.

"Based on the information, your heart health outlook appears reassuring, indicating no heart disease at this time. It's great that your heart's electrical activity is normal and you don't experience chest discomfort with exercise.\n\nWe noticed your cholesterol level of 210 is a little higher than ideal, and your fasting blood sugar of 100 is at the upper end of the normal range. It's also a good idea to ensure your blood pressure is regularly checked and within a healthy range.\n\nFocusing on a balanced diet, regular physical activity, and routine check-ups with your doctor can help keep your heart healthy and manage these numbers effectively. Your doctor can offer personalized advice and support."

## Evaluation Summary

### Qualitative Metrics

**1. Edge-Case Handling**
| Metric | Evaluation |
|---|---|
| Edge-Case Handling |  Zero-shot template does not handle medically nonsensical input robustly. Although the patient record included a RestingBP value of 0, which is clinically impossible, the generated overview did not identify it as invalid and instead produced a generally reassuring interpretation. This suggests that the template follows the overall prediction and the surrounding feature pattern more strongly than it checks whether each individual value is medically plausible. As a result, the template may still generate fluent and calm responses even when the input contains impossible measurements, which is a limitation for safety and reliability in unusual or corrupted cases.|

---

**2. Instruction Following**


| Sub-category | Description |
|---|---|
| Personalization | Across the test cases, the explanations change according to the combination of feature values, not only according to whether the model predicted heart disease or no heart disease. In Test Case 1, the response reflects a risk-oriented interpretation by emphasizing concerning indicators such as elevated blood pressure, high cholesterol, increased fasting blood sugar, exercise-related chest discomfort, and abnormal activity-related findings. In Test Case 2, the output shifts to a calmer and more reassuring explanation, focusing on healthy values and the absence of major warning signs. In Test Case 3, the response remains different from the fully healthy case by acknowledging that the overall prediction is negative while still pointing out borderline values that deserve attention, which shows a more tailored and preventive interpretation. In the edge case, although the system did not correctly handle the medically impossible value, the generated explanation was still influenced by that case’s surrounding clinical pattern rather than repeating a fixed message from the earlier examples. This indicates that the template demonstrates meaningful personalization, even though its handling of invalid extreme inputs still needs improvement.|
| Context | Across the four test cases, the responses remained consistently within the heart disease context. Each output stayed focused on cardiovascular indicators such as blood pressure, cholesterol, blood sugar, chest pain, and exercise-related findings, which shows strong contextual alignment with the purpose of the system.  |
| Safety — No Diagnosis | In terms of safety, the responses did not follow the no-diagnosis instruction consistently. In Test Case 1, the response remained relatively safe because it said “your heart may need some extra care,” which is cautionary and interpretive rather than directly diagnostic. However, in Test Case 2, the phrase “your heart health appears to be in a good state” still moves toward a health judgment, and in Test Case 3, “indicates no signs of heart disease” is more clearly diagnostic because it states the absence of disease rather than interpreting the assessment cautiously. The same issue appears in Test Case 4 with “indicating no heart disease at this time,” so overall the template did not preserve the safety requirement well, because several responses framed the output as a disease conclusion instead of a non-diagnostic health overview. |


### Quantative Metrics

In [111]:
# Output preparation
zero_shot_output={
    "TC1 — Heart Disease Detected": """Based on your information, it suggests your heart health needs a closer look. We noticed your blood pressure is a bit higher than ideal, and your cholesterol and blood sugar levels are elevated. You also mentioned experiencing some chest discomfort during exercise, and tests showed your heart working harder during physical activity.

    These findings are important to address. We strongly recommend discussing this overview with your doctor soon. They can help you understand these points better and create a personalized plan to support your heart health, possibly including lifestyle adjustments and other steps.""",

    "TC2 — No Heart Disease": """Your current health profile is very encouraging, suggesting that you do not show signs of heart disease. Your blood pressure, cholesterol, and blood sugar levels are all looking good. This is a wonderful foundation for your heart health. Continue with your healthy habits to keep your heart strong.""",

    "TC3 — No Heart Disease with Borderline Values": """Your recent heart health assessment is reassuring, indicating you do not show signs of heart disease at this time.

    To support your long-term well-being, we noticed a few areas that could benefit from your attention. Your blood pressure and cholesterol levels are a little higher than ideal. Additionally, your fasting blood sugar is at a level where it's good to keep an eye on it.

    Embracing a healthy lifestyle with a balanced diet and regular activity is a wonderful way to care for your heart. We recommend discussing these points with your doctor for personalized guidance.""",

    "TC4 — Edge Case": """Based on the information, your heart health outlook appears reassuring, indicating no heart disease at this time. It's great that your heart's electrical activity is normal and you don't experience chest discomfort with exercise.

    We noticed your cholesterol level of 210 is a little higher than ideal, and your fasting blood sugar of 100 is at the upper end of the normal range. It's also a good idea to ensure your blood pressure is regularly checked and within a healthy range.

    Focusing on a balanced diet, regular physical activity, and routine check-ups with your doctor can help keep your heart healthy and manage these numbers effectively. Your doctor can offer personalized advice and support."""
}

#### 1. Flesch-Kincaid Readability Score

In [112]:
score= evaluate_readability(zero_shot_output)

Flesch-Kincaid Readability Scores:
--------------------------------------------------
TC1 — Heart Disease Detected: 48.28
TC2 — No Heart Disease: 67.76
TC3 — No Heart Disease with Borderline Values: 54.88
TC4 — Edge Case: 47.29

Template Average: 54.55


| Test Case | Score       |
|---|-------------|
| TC1 — Heart Disease Detected | 59.29       |
| TC2 — No HD: All Healthy | 60.1        |
| TC3 — No HD: Borderline | 53.05       |
| TC4 — Edge Case | 47.29       |
| **Template Average** | **54.93**   |
| **Target** | **70 — 80** |

The readability results suggest that the zero-shot outputs are not fully suitable for an anxious user who needs fast to process language. With TC1 = 59.29, TC2 = 60.1, TC3 = 53.05, and TC4 = 53.93, all four cases fall below the more appropriate target range of 70–80 for this user type (Diaz, 2024). Overall, although some responses are more readable than others, the template average of 54.93 shows that the outputs remain denser and harder to absorb than what would be ideal for a user seeking immediate reassurance, simple meaning, and quick guidance.

> Diaz, K. (2024). Designing for anxious users: A psychology-driven approach.
> https://medium.com/design-bootcamp/designing-for-anxious-users-a-psychology-driven-approach-f8df58edad06

#### 2. Domain Keyword Density

In [113]:
score = evaluate_keyword_density(zero_shot_output)

Domain Keyword Density:
--------------------------------------------------
TC1 — Heart Disease Detected:
  Total Words       : 93
  Domain Keywords   : 5
  Density           : 5.38%

TC2 — No Heart Disease:
  Total Words       : 48
  Domain Keywords   : 3
  Density           : 6.25%

TC3 — No Heart Disease with Borderline Values:
  Total Words       : 96
  Domain Keywords   : 3
  Density           : 3.12%

TC4 — Edge Case:
  Total Words       : 114
  Domain Keywords   : 6
  Density           : 5.26%

Template Average Density: 5.0%


| Test Case | Domain Keywords | Total Words | Density   |
|---|-----------------|-------------|-----------|
| TC1 — Heart Disease Detected | 5               | 96          | 5.21%     |
| TC2 — No HD: All Healthy | 2               | 54          | 3.7%      |
| TC3 — No HD: Borderline | 3               | 108         | 2.78%     |
| TC4 — Edge Case | 6               | 114         | 4.26%     |
| **Template Average** |                 |             | **4.24%** |

The observed domain keyword density is suitable for this use case, with an average density of 4.24%. This can be considered a positive result because the responses are intended for an anxious user, where overly dense medical terminology may increase stress, confusion, or cognitive overload. By keeping the average density relatively low, the language remains medically relevant without becoming too technical or overwhelming. This supports clearer communication, which is especially important when explaining cardiovascular information to users who may already feel worried about their condition.

#### 3. Medical Accuracy


| # | Test Case | Clinical Claim                                                           | Correct |
|---|-----------|--------------------------------------------------------------------------|---------|
| 1 | TC1       | Your blood pressure is a bit higher than ideal.                       |  No     |
| 2 | TC1       | Cholesterol levels is elevated.                         |  No     |
| 3 | TC1       | Blood sugar levels is elevated.                         | No      |
| 4 | TC2       | Your blood pressure levels is all looking good.   | Yes     |
| 5 | TC2       | Your cholesterol levels is all looking good.   | Yes     |
| 6 | TC2       | Your blood sugar levels is all looking good.   | Yes     |
| 7 | TC3       | Your blood pressure levels is a little higher than ideal.       | Yes     |
| 8 | TC3       | Your cholesterol levels is a little higher than ideal.       | Yes     |
| 9 | TC3       | Fasting blood sugar is at a level where it's good to keep an eye on it | Yes     |
| 10 | TC4       | Cholesterol level of 210 is a little higher than ideal                   | Yes     |
| 11 | TC4       | Fasting blood sugar of 100 is at the upper end of the normal range       | No      |

---

| Test Case | Correct Claims | Total Claims | Accuracy Rate |
|---|----------------|--------------|---------------|
| TC1 | 0              | 2            | 0%           |
| TC2 | 3              | 3            | 100%          |
| TC3 | 3              | 3            | 100%        |
| TC4 | 1              | 2            | 50%           |
| **Template Average** |                |              | **62.5%**    |
---

#### 4. Passive Voice Rate

In [115]:
score = evaluate_passive_voice(zero_shot_output)

Passive Voice Rate:
--------------------------------------------------
TC1 — Heart Disease Detected:
  Total Sentences  : 6
  Passive Sentences: 1
  Passive Rate     : 16.67%

TC2 — No Heart Disease:
  Total Sentences  : 4
  Passive Sentences: 0
  Passive Rate     : 0.0%

TC3 — No Heart Disease with Borderline Values:
  Total Sentences  : 6
  Passive Sentences: 0
  Passive Rate     : 0.0%

TC4 — Edge Case:
  Total Sentences  : 6
  Passive Sentences: 1
  Passive Rate     : 16.67%

Template Average Passive Voice Rate: 8.34%


| Test Case | Passive Sentences | Total Sentences | Passive Rate |
|---|-------------------|-----------------|--------------|
| TC1 — Heart Disease Detected | 0                 | 6               | 0%           |
| TC2 — No HD: All Healthy | 0                 | 3               | 0%           |
| TC3 — No HD: Borderline | 1                 | 6               | 16.67%       |
| TC4 — Edge Case | 1                 | 6               | 16.67%       |
| **Template Average** |                   |                 | **8.34%**    |

The passive voice results are favorable overall, with an average rate of 8.34% across the test cases. This suggests that most of the generated responses were written in a direct and active style rather than in a detached or overly formal way. That is especially suitable in this project, since the explanations are intended for anxious users who need language that feels clear, supportive, and easy to follow. Although passive voice appeared in TC3 and TC4 at 16.67%, it was completely absent in TC1 and TC2, so the overall pattern still shows that the template generally maintains an active tone rather than sounding like a rigid clinical report.

## Assumptions and Limitations

### Assumptions

**1.  Anxious Patient Profile**
The template assumes that the patient is likely to feel worried when receiving health-related information and therefore needs immediate reassurance, simple meaning, and clear guidance. The tone, level of detail, and wording were calibrated around this assumption. If used for patients who prefer more detailed, technical, or explanation-heavy responses, the template may feel too brief or overly simplified.


**2. Plain-Language Communication Preference**
The template assumes that the patient has no medical background and benefits most from plain, patient-friendly language rather than technical terminology. This assumption shaped the instruction to avoid jargon, abbreviations, and dense explanation. If the user has higher health literacy or expects more clinically detailed interpretation, the output may not fully meet their informational needs.


---

### Limitations


**1. Limited Context from System Instructions Alone**
Because the template relies only on system instructions and direct input values, the model must infer the desired structure, emphasis, and balance of explanation without any examples or deeper contextual guidance. As a result, the output may vary across cases in tone, detail, or phrasing even when the same style is intended. This can reduce consistency, especially when patient profiles contain mixed or borderline indicators.

**2. Risk of Diagnostic Drift**
Although the template instructs the model not to provide a diagnosis, the generated wording may still drift toward diagnostic phrasing in some cases. Expressions such as indicating the presence or absence of heart disease can sound more conclusive than intended, even when the goal is only to interpret the assessment result. This makes the template vulnerable to crossing from supportive explanation into language that feels clinically definitive.

**3. Dependence on the Model’s General Medical Knowledge**
As presented in the evaluation of medical accuracy, the template assumes the model can interpret the provided values correctly using its internal knowledge of cardiovascular health. However, that knowledge is not guaranteed to be perfectly aligned with the exact clinical references or simplified thresholds used in the project evaluation. This creates room for subtle mismatches between the generated wording and the reference standards used for assessment.

**4. Emotional Overcorrection Risk**
Because the template is designed to sound reassuring and gentle, it may sometimes soften concern too much in an effort to avoid scaring the user. In some cases, this may reduce the urgency of medically important findings. So the same feature that makes the template feel safe can also make it less forceful than needed.



# Comparative Analysis Across The 4 Templates


## Qualitative Comparison

### Edge Case Handling

Across the four templates, only Template 2 successfully identified and
flagged the implausible RestingBP value of 0 mmHg as clinically
impossible. Templates 1, 3, and 4 all proceeded to generate
confident responses without acknowledging the invalid input.

### Instruction Following



**Personalisation**

All four templates successfully adapted their responses to the specific
patient's clinical values rather than producing a generic label-driven
response. Across all templates, no two responses were identical despite
sharing the same input structure, confirming that the output is
consistently driven by individual values. However the templates differ
in how they represent the personalisation aspect. Template 2 achieves
personalisation through two layers by explicitly referencing specific
numerical values in both the per-value table and the interpretation
paragraph. Templates 1, 3, and 4 achieve personalisation by directly
describing the clinical implications of the patient's values without
necessarily repeating the exact numbers in every instance. Both
approaches reflect meaningful personalisation as the response is always
shaped by the individual patient's clinical profile rather than the
prediction label alone.

**Context**

All four templates maintained the clinical and patient context
established in the prompt consistently across all test cases. Responses
remained focused on cardiovascular test results and patient-facing
insights without drifting into unrelated or inappropriate territory.


**Safety — No Diagnosis**

The templates showed notable variation in their adherence to this
constraint. Templates 1, 2, and 3 consistently avoided stating or
implying a diagnosis, framing all findings as possibilities using
cautious probabilistic language such as "suggests" and "low probability".
Template 4 was the weakest in this area, with some responses framing findings as confirmed conclusions rather
than possibilities. Phrases such as "indicating you do not show signs
of heart disease at this time" and "indicating no heart disease at
this time" present findings as confirmed facts rather than
probabilistic assessments, which represents a notable violation of
the safety constraint.

## Quantitative Comparison


| Metric | Generated Knowledge | Chain-of-Thought | Few-Shot | Zero-Shot |
|---|---|---|---|---|
| FK Readability Score | 39.52 | 42.66 | 20.66 | 54.55 |
| Domain Keyword Density | 14.21% | 8.36% | 5.96% | 5% |
| Medical Accuracy | 70.84% | 80% | 87.5% | 62.5% |
| Passive Voice Rate | 26.46% | 14.57% | 31.25% | 8.34% |

### Interpretation of Quantitative Results


**Flesch-Kincaid Readability Score:**

The readability scores must be interpreted relative to each template's intended user rather than against a single universal target.
Zero-Shot achieves the highest score at 54.55, consistent with its goal of producing brief, accessible output for an anxious patient with no medical background. This alignment is a positive finding. 
Few-Shot scores the lowest at 20.66, which is a significant concern. Even though Few-Shot targets a returning patient, a returning patient is not a medical expert, they are simply just familiar with the system's format. A readability score this low suggests the responses are linguistically denser than what any non-expert patient should reasonably be expected to process, regardless of their familiarity with the application.
A particularly notable finding emerges when comparing Generated Knowledge and Chain-of-Thought. Both score similarly, 39.52 and 42.66 respectively, a difference of only approximately 3 points. However, Generated Knowledge targets a first-time patient with no medical background, while Chain-of-Thought targets a health-literate patient who has explicitly chosen detailed reasoning. The fact that the template designed for a medically inexperienced user is less readable than the one designed for a health-literate user is a suspicious pattern.


**Domain Keyword Density:**

Chain-of-Thought achieves the highest density at 8.36%, which is appropriate for its health-literate intended user who expects clinical engagement with their specific values and is comfortable with cardiovascular terminology. Zero-Shot achieves the lowest density at 5%, which aligns well with its objective of producing accessible, low-jargon output for an anxious patient who needs immediate clarity rather than clinical depth. The overall ordering maps logically onto the expected information needs of each user type, from the most emotionally vulnerable to the most clinically engaged.
The most significant observation, however, is the relationship between Generated Knowledge and its intended user. At 6.75%, Generated Knowledge sits above Few-Shot and Zero-Shot in density, which raises a concern given that it targets a first-time patient with no medical background who is the least equipped of all four user types to process clinical terminology. This pattern can be explained by how Generated Knowledge works: the technique instructs the model to first gather relevant clinical knowledge about the patient's values before writing the final response. This internal knowledge generation step is what makes the output clinically grounded, but it also causes the model to introduce cardiovascular domain terms that a first time patient may not be equipped to process, even when those terms are explained in context. This makes the template more complex than what a first-time patient needs, which may make it harder to understand



**Medical Accuracy:**

A notable pattern appears when comparing Few-Shot and Generated Knowledge. Few-Shot achieves a highest average accuracy 87.5%, even though Generated Knowledge is designed to retrieve relevant clinical information before responding, which should support accuracy.
However, this average for Few-Shot hides an important issue. One test case scored 0% accuracy, meaning the template can perform very well in some cases but fail completely in others, depending on how similar the input is to the examples it was given. In contrast, Generated Knowledge produces more consistent results because it adapts to each patient’s data instead of relying on fixed examples. Therefore, Few-Shot’s higher average accuracy may be misleading and less reliable in real-world use. Chain-of-Thought sits in between at 80% and this is a meaningful result. Its accuracy is stable across cases, and its step by step reasoning helps reduce mistakes by making the model process each input clearly. This also explains its higher use of domain terms, as the reasoning process naturally introduces more clinical terminology.

**Passive Voice Rate:**

Few-Shot's passive rate of 31.25% is the highest across all templates, and this is likely a direct reflection of the example responses provided in the prompt. Looking at the examples, phrases such as "These factors are commonly associated with a lower cardiovascular risk profile" and "These patterns are often associated with higher cardiovascular risk" are structurally passive and appear repeatedly across the sample outputs. Since Few-Shot learns its tone and phrasing directly from these examples, the model replicates this formal, associative style rather than adopting a more direct patient-facing voice. The passive rate is therefore not a limitation of the technique in isolation, but a consequence of the specific examples chosen.
Generated Knowledge is ranked second highest which is considered the most important observation in this metric. The relationship between Generated Knowledge's high passive rate at 26.46% and the mechanism behind the technique is because the model is first instructed to retrieve and present clinical knowledge before addressing the patient, it defaults to fact-presenting language, using structures that report information rather than direct it toward someone. 
Chain-of-Thought's third place ranking at 14.57% is actually a positive finding rather than a concern. The technique naturally produces a two-phase structure: passive constructions appear in the reasoning phase where concepts are being explained, then shift to active voice when the model addresses the patient directly. This balance is intentional and appropriate for a health-literate user who expects both clinical explanation and direct guidance.
Zero-Shot's lowest rate at 8.34% is its strongest result across all metrics. For an anxious patient, active language is not a stylistic preference but a functional requirement. Direct framing reduces the emotional distance between the response and the reader, which is precisely what this user type needs.


### Limitations

Following the selection of Template 2 as the chosen approach,
there are several system-level limitations that must be taken into
consideration before deployment. These limitations are not specific
to the chain-of-thought technique itself but rather reflect broader
constraints of the system that remain relevant regardless of which
template is selected.

**1. Dependency on the XGBoost Prediction Label**
The generated insight is anchored to the XGBoost prediction label
as its primary input. This means that if the underlying model
produces an incorrect prediction, the chain-of-thought reasoning
will be built around a misleading result, and the detailed per-value
analysis will be framed incorrectly regardless of how thorough the
reasoning steps are.

**2. No Systematic Input Validation Mechanism**
Although Template 2 successfully detected the implausible RestingBP
value of 0 mmHg in the edge case, this detection was a result of
the chain-of-thought structure rather than a deliberate validation
step. The system does not include a built-in input validation
mechanism, meaning that detection of implausible values cannot be
guaranteed consistently across all possible invalid inputs.

**3. Non-Deterministic Output**
The template relies on a large language model whose outputs are
non-deterministic. Two identical patient inputs may produce slightly
different reasoning steps and insights across runs, which could
affect consistency and reliability in a clinical deployment context.

**4. No External Clinical Verification**
The chain-of-thought reasoning relies entirely on the model's
internal knowledge. There is no external verification mechanism to
confirm that the clinical statements produced in each reasoning
step are factually correct, which means inaccurate statements may
appear without being detected.

**5. Single Language Support**
The template is designed and evaluated in English only, which limits
its applicability in multilingual hospital settings.

### Ethical Considerations

Deploying an AI-driven cardiovascular insight system in a hospital
setting raises several ethical considerations that must be carefully
addressed to ensure the system operates responsibly and in the best
interest of the patient.

**1. Risk of Misinterpretation**
Although Template 2 consistently frames findings as probabilistic
assessments rather than confirmed diagnoses, there is a risk that
patients may still interpret the generated insight as a medical
conclusion. A patient with no medical background may not fully
understand the distinction between a probabilistic assessment and
a clinical diagnosis, which could lead to unnecessary anxiety or
false reassurance.

**2. Patient Autonomy and Informed Consent**
The system generates personalised health insights based on the
patient's clinical data using an artificial intelligence model.
Patients should be fully informed that the insight is generated
by an AI system and not by a licensed medical professional.
Transparency about the nature and limitations of the system is
essential to preserve patient autonomy and trust.

**3. Accountability and Clinical Responsibility**
Since the generated insight is anchored to the XGBoost prediction
label, any inaccuracy in the underlying model will directly
propagate into the patient-facing output. It is important to
clearly define who holds clinical responsibility for the insights
produced by the system, particularly in cases where the output
may influence a patient's decision to seek or delay medical care.

**4. Health Equity**
The system is currently designed and evaluated in English only
and assumes a hospital mobile application deployment context.
This may disadvantage patients who do not speak English or who
do not have access to mobile technology, raising concerns about
equitable access to the system's benefits across different
patient populations.

**5. Data Privacy**
The system processes sensitive patient clinical data including
blood pressure, cholesterol, fasting blood sugar, and ECG
findings. Appropriate data protection measures must be in place
to ensure that patient data is handled securely and in compliance
with relevant healthcare data privacy regulations.

**6. Over-Reliance on AI**
There is a risk that patients or healthcare providers may
over-rely on the system's output without seeking further
professional evaluation. Template 2 is designed as a
decision-support tool and should never be used as a substitute
for professional medical advice, diagnosis, or treatment.